# 91_pdf_segment_validation_audit — block/turn/retrieval segment 검증 감사

**목적**: `02_pdf_crawler.ipynb`(PDF 수집)와 `90_pdf_text_extraction_pilot.ipynb`(추출/분절 파일럿)의
결과를 **사실로 가정하지 않고 처음부터 재검증**한다. 이 노트북은 canonical SSOT 산출물을 만들지 않는다 —
`outputs/segment_validation_audit/` 아래의 모든 산출물은 감사 전용이며, `blocks.parquet`/`speaker_turns.parquet`/
`retrieval_segments.parquet` 같은 canonical 이름으로는 아무것도 저장하지 않는다.

각 섹션은 위에서부터 순서대로 실행하면서 수치·오류 행·원문 샘플을 직접 확인하고,
다음 판단을 위 셀 출력에서 바로 얻을 수 있도록 구성했다. 최종 판정은 단순 PASS/FAIL이 아니라
`READY_FOR_CANONICAL_SEGMENT_EXPORT` / `CONDITIONAL_READY` / `NOT_READY` 중 하나다.


## 00. 실행 계약 · 설정 · 출력 정책

**목적**: 이 노트북의 실행 모드, 허용/금지 범위, 출력 경로, 환경 정보를 고정하고 기록한다.
**입력**: 없음(환경 자체를 조사).
**출력**: 설정 표, 환경 정보, `[SECTION 00 RESULT]` 요약.
**가능한 실패**: 의존성 미설치(자동 설치 금지 — SKIP/FAIL로만 기록), 디스크 공간 부족, 출력 폴더 생성 실패.


In [1]:
import sys, os, subprocess, platform, json, re, hashlib, glob, unicodedata, logging
import shutil
from pathlib import Path
from datetime import datetime, timezone, timedelta
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import nbformat as nbf

PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE")
NOTEBOOK_PATH = PROJECT_ROOT / "91_pdf_segment_validation_audit.ipynb"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "segment_validation_audit"
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_PATH = LOG_DIR / "segment_validation_audit.log"

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "full_audit"
SOURCE_PRIORITY = ["canonical_parquet", "pilot_cache", "rebuild_from_pdf"]
REBUILD_ONLY_IF_NEEDED = True
ALLOW_REBUILD_FROM_PDF = True

RUN_PDF_INVENTORY_AUDIT = True
RUN_PAGE_AUDIT = True
RUN_BLOCK_AUDIT = True
RUN_TURN_AUDIT = True
RUN_SEGMENT_BUILD_IN_MEMORY = True
RUN_SEGMENT_AUDIT = True
RUN_TARGET_COMPATIBILITY_AUDIT = True
RUN_RETRIEVAL_SMOKE_TEST = True
RUN_AUDIT_EXPORT = True

RUN_OCR = False
RUN_EMBEDDING = False
RUN_LLM = False
RUN_SQLITE = False
ALLOW_CANONICAL_EXPORT = False

RETRIEVAL_BOUNDARY_POLICY = "omit_missing_neighbor"
RANDOM_SEED = 20260721
MAX_DISPLAY_ROWS = 20

SMOKE_TARGETS_PER_YEAR = 5
SMOKE_TOP_K = 10

CHAR_NGRAM_RANGE = (3, 5)
WORD_NGRAM_RANGE = (1, 2)
CHAR_SCORE_WEIGHT = 0.65
WORD_SCORE_WEIGHT = 0.35

TARGET_CHUNK_MIN = 800
TARGET_CHUNK_MAX = 1500
CHUNK_OVERLAP = 200

rng = np.random.default_rng(RANDOM_SEED)

logger = logging.getLogger("segment_validation_audit")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
logger.addHandler(_sh)

RUN_STARTED_UTC = datetime.now(timezone.utc)
RUN_STARTED_KST = RUN_STARTED_UTC.astimezone(timezone(timedelta(hours=9)))

def _git(*args):
    try:
        return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True,
                               text=True, check=True).stdout.strip()
    except Exception as e:
        return f"UNAVAILABLE({e})"

GIT_BRANCH = _git("branch", "--show-current")
GIT_HEAD = _git("rev-parse", "HEAD")

def _module_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown"), True
    except Exception:
        return None, False

_pymupdf_ver, _pymupdf_ok = _module_version("fitz")
if _pymupdf_ok:
    import fitz
    _pymupdf_ver = getattr(fitz, "pymupdf_version", _pymupdf_ver)
_sklearn_ver, _sklearn_ok = _module_version("sklearn")

disk = shutil.disk_usage(PROJECT_ROOT)

SETTINGS_TABLE = {
    "RUN_MODE": RUN_MODE, "SOURCE_PRIORITY": SOURCE_PRIORITY,
    "REBUILD_ONLY_IF_NEEDED": REBUILD_ONLY_IF_NEEDED, "ALLOW_REBUILD_FROM_PDF": ALLOW_REBUILD_FROM_PDF,
    "RUN_PDF_INVENTORY_AUDIT": RUN_PDF_INVENTORY_AUDIT, "RUN_PAGE_AUDIT": RUN_PAGE_AUDIT,
    "RUN_BLOCK_AUDIT": RUN_BLOCK_AUDIT, "RUN_TURN_AUDIT": RUN_TURN_AUDIT,
    "RUN_SEGMENT_BUILD_IN_MEMORY": RUN_SEGMENT_BUILD_IN_MEMORY, "RUN_SEGMENT_AUDIT": RUN_SEGMENT_AUDIT,
    "RUN_TARGET_COMPATIBILITY_AUDIT": RUN_TARGET_COMPATIBILITY_AUDIT,
    "RUN_RETRIEVAL_SMOKE_TEST": RUN_RETRIEVAL_SMOKE_TEST, "RUN_AUDIT_EXPORT": RUN_AUDIT_EXPORT,
    "RUN_OCR": RUN_OCR, "RUN_EMBEDDING": RUN_EMBEDDING, "RUN_LLM": RUN_LLM, "RUN_SQLITE": RUN_SQLITE,
    "ALLOW_CANONICAL_EXPORT": ALLOW_CANONICAL_EXPORT,
    "RETRIEVAL_BOUNDARY_POLICY": RETRIEVAL_BOUNDARY_POLICY, "RANDOM_SEED": RANDOM_SEED,
    "MAX_DISPLAY_ROWS": MAX_DISPLAY_ROWS,
    "SMOKE_TARGETS_PER_YEAR": SMOKE_TARGETS_PER_YEAR, "SMOKE_TOP_K": SMOKE_TOP_K,
    "CHAR_NGRAM_RANGE": CHAR_NGRAM_RANGE, "WORD_NGRAM_RANGE": WORD_NGRAM_RANGE,
    "CHAR_SCORE_WEIGHT": CHAR_SCORE_WEIGHT, "WORD_SCORE_WEIGHT": WORD_SCORE_WEIGHT,
    "TARGET_CHUNK_MIN": TARGET_CHUNK_MIN, "TARGET_CHUNK_MAX": TARGET_CHUNK_MAX, "CHUNK_OVERLAP": CHUNK_OVERLAP,
}

print("=== SETTINGS ===")
for k, v in SETTINGS_TABLE.items():
    print(f"{k}: {v}")

print()
print("=== ENVIRONMENT ===")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"NOTEBOOK_PATH: {NOTEBOOK_PATH}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"run_started_utc: {RUN_STARTED_UTC.isoformat()}")
print(f"run_started_kst: {RUN_STARTED_KST.isoformat()}")
print(f"git_branch: {GIT_BRANCH}")
print(f"git_head: {GIT_HEAD}")
print(f"python_version: {sys.version.split()[0]}")
print(f"pandas_version: {pd.__version__}")
print(f"numpy_version: {np.__version__}")
print(f"pymupdf_available: {_pymupdf_ok} version={_pymupdf_ver}")
print(f"sklearn_available: {_sklearn_ok} version={_sklearn_ver}")
if not _sklearn_ok:
    logger.warning("scikit-learn NOT installed -- will NOT auto-install. "
                    "Section 15 substitutes a manual numpy-based char/word TF-IDF + cosine implementation.")
print(f"disk_total_gb: {disk.total/1e9:.1f}  disk_free_gb: {disk.free/1e9:.1f}")

import resource
_maxrss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print(f"process_maxrss_mb_so_far: {_maxrss_mb:.1f}")

assert not ALLOW_CANONICAL_EXPORT, "ALLOW_CANONICAL_EXPORT must stay False in this audit notebook"

print()
print("[SECTION 00 RESULT]")
print("status: PASS")
print(f"run_mode: {RUN_MODE}")
print("canonical_export: DISABLED")
print(f"output_root: {OUTPUT_ROOT}")
print("next: repository inventory")


WARNING scikit-learn NOT installed -- will NOT auto-install. Section 15 substitutes a manual numpy-based char/word TF-IDF + cosine implementation.


=== SETTINGS ===
RUN_MODE: full_audit
SOURCE_PRIORITY: ['canonical_parquet', 'pilot_cache', 'rebuild_from_pdf']
REBUILD_ONLY_IF_NEEDED: True
ALLOW_REBUILD_FROM_PDF: True
RUN_PDF_INVENTORY_AUDIT: True
RUN_PAGE_AUDIT: True
RUN_BLOCK_AUDIT: True
RUN_TURN_AUDIT: True
RUN_SEGMENT_BUILD_IN_MEMORY: True
RUN_SEGMENT_AUDIT: True
RUN_TARGET_COMPATIBILITY_AUDIT: True
RUN_RETRIEVAL_SMOKE_TEST: True
RUN_AUDIT_EXPORT: True
RUN_OCR: False
RUN_EMBEDDING: False
RUN_LLM: False
RUN_SQLITE: False
ALLOW_CANONICAL_EXPORT: False
RETRIEVAL_BOUNDARY_POLICY: omit_missing_neighbor
RANDOM_SEED: 20260721
MAX_DISPLAY_ROWS: 20
SMOKE_TARGETS_PER_YEAR: 5
SMOKE_TOP_K: 10
CHAR_NGRAM_RANGE: (3, 5)
WORD_NGRAM_RANGE: (1, 2)
CHAR_SCORE_WEIGHT: 0.65
WORD_SCORE_WEIGHT: 0.35
TARGET_CHUNK_MIN: 800
TARGET_CHUNK_MAX: 1500
CHUNK_OVERLAP: 200

=== ENVIRONMENT ===
PROJECT_ROOT: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE
NOTEBOOK_PATH: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/91_pdf_segment_validation_audi

## 01. Repository 및 파일 인벤토리

**목적**: 이 저장소에 실제로 어떤 산출물(노트북/parquet/json/csv/md/pdf)이 존재하는지 전수 조사하고,
block/turn source 후보를 찾는다. 파일 내용을 전부 출력하지 않고 목록·크기·해시만 기록한다.
**입력**: `PROJECT_ROOT` 하위 전체 파일 트리(단, `.git/`, `outputs/segment_validation_audit/` 자기 자신은 제외).
**출력**: 파일 인벤토리 DataFrame, source 후보 목록, 충돌/의심 파일 목록.
**가능한 실패**: 매우 큰 파일로 인한 해시 지연, 이름은 같으나 경로가 다른 동명 파일 충돌.


In [2]:
def sha256_of_file(path: Path, max_bytes: int | None = None) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        read = 0
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
            read += len(chunk)
            if max_bytes and read >= max_bytes:
                break
    return h.hexdigest()

ROLE_BY_SUFFIX = {
    ".ipynb": "notebook", ".parquet": "parquet", ".json": "json_manifest",
    ".csv": "csv", ".md": "markdown_report", ".pdf": "pdf", ".xlsx": "excel_registry",
}
EXCLUDE_DIR_PARTS = {".git", "outputs/segment_validation_audit", ".ipynb_checkpoints"}

def classify_role(rel_path: str, suffix: str) -> str:
    base = ROLE_BY_SUFFIX.get(suffix, "other")
    low = rel_path.lower()
    if base == "parquet":
        if "control_registry" in low: return "registry_parquet"
        if "download_log" in low or "crawl_quality" in low: return "crawl_meta_parquet"
        if any(k in low for k in ("blocks", "block_df")): return "block_source_candidate"
        if any(k in low for k in ("speaker_turns", "turn_df", "turns_pilot")): return "turn_source_candidate"
        if any(k in low for k in ("pages", "page_df")): return "page_source_candidate"
        if "retrieval_segments" in low or "segment_df" in low: return "segment_source_candidate"
        if "marked_issue_mapping" in low: return "target_parquet"
        return "parquet_other"
    if base == "csv" and "marked_issue_mapping" in low:
        return "target_csv"
    return base

print("scanning file tree ...")
EXCLUDE_ABS = [str(PROJECT_ROOT / p) for p in EXCLUDE_DIR_PARTS]

def is_excluded(p: Path) -> bool:
    sp = str(p)
    return any(sp.startswith(ex) for ex in EXCLUDE_ABS)

inventory_rows = []
SUFFIXES_OF_INTEREST = {".ipynb", ".parquet", ".json", ".csv", ".md", ".pdf", ".xlsx"}
for p in PROJECT_ROOT.rglob("*"):
    if p.is_dir() or is_excluded(p):
        continue
    if p.suffix.lower() not in SUFFIXES_OF_INTEREST:
        continue
    rel = str(p.relative_to(PROJECT_ROOT))
    stat = p.stat()
    try:
        digest = sha256_of_file(p) if stat.st_size < 50_000_000 else "SKIPPED_TOO_LARGE"
    except Exception as e:
        digest = f"HASH_ERROR:{e}"
    inventory_rows.append({
        "file_path": rel, "file_type": p.suffix.lower().lstrip("."),
        "file_size": stat.st_size,
        "modified_at": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
        "sha256": digest, "role_candidate": classify_role(rel, p.suffix.lower()),
    })

file_inventory_df = pd.DataFrame(inventory_rows).sort_values("file_path").reset_index(drop=True)
print(f"files inventoried: {len(file_inventory_df)}")


scanning file tree ...
files inventoried: 96


In [3]:
active_kernel_count = 0
try:
    ps_out = subprocess.run(["ps", "-eo", "cmd"], capture_output=True, text=True).stdout
    active_kernel_count = sum(1 for line in ps_out.splitlines() if "ipykernel_launcher" in line)
except Exception as e:
    logger.warning(f"could not count active kernels: {e}")

pdf_rows = file_inventory_df[file_inventory_df["role_candidate"] == "pdf"]
notebook_rows = file_inventory_df[file_inventory_df["role_candidate"] == "notebook"]
parquet_rows = file_inventory_df[file_inventory_df["file_type"] == "parquet"]
json_rows = file_inventory_df[file_inventory_df["file_type"] == "json"]
csv_rows = file_inventory_df[file_inventory_df["file_type"] == "csv"]
md_rows = file_inventory_df[file_inventory_df["file_type"] == "md"]

source_candidates = file_inventory_df[file_inventory_df["role_candidate"].isin([
    "block_source_candidate", "turn_source_candidate", "page_source_candidate", "segment_source_candidate",
])]

# duplicate basenames at different paths = potential conflicts
file_inventory_df["basename"] = file_inventory_df["file_path"].map(lambda x: Path(x).name)
dup_basenames = file_inventory_df[file_inventory_df.duplicated("basename", keep=False)].sort_values("basename")

# notebooks whose stored outputs look stale relative to their own mtime vs referenced data mtimes: flag any
# notebook with 0 executed cells as a "stale/never-run" candidate (informational only, no execution here)
stale_notebook_candidates = []
for _, row in notebook_rows.iterrows():
    try:
        nb_check = nbf.read(PROJECT_ROOT / row["file_path"], as_version=4)
        code_cells = [c for c in nb_check.cells if c.cell_type == "code"]
        n_exec = sum(1 for c in code_cells if c.get("execution_count") is not None)
        if len(code_cells) > 0 and n_exec == 0:
            stale_notebook_candidates.append(row["file_path"])
    except Exception as e:
        logger.warning(f"could not parse notebook {row['file_path']}: {e}")

print("=== git / process context ===")
print(f"git_branch={GIT_BRANCH}  git_head={GIT_HEAD}")
print(f"active_ipykernel_processes={active_kernel_count}")

print()
print("=== inventory counts ===")
print(f"pdf={len(pdf_rows)}  notebooks={len(notebook_rows)}  parquet={len(parquet_rows)}  "
      f"json={len(json_rows)}  csv={len(csv_rows)}  md={len(md_rows)}")

print()
print("=== source candidates found ===")
if len(source_candidates):
    print(source_candidates[["file_path","role_candidate","file_size"]].to_string(index=False))
else:
    print("(none found -- expect SOURCE_MODE=rebuild_from_pdf in section 03)")

print()
print("=== conflicting (duplicate basename) files ===")
print(dup_basenames[["file_path","basename","file_size"]].to_string(index=False) if len(dup_basenames) else "(none)")

print()
print("=== stale/never-run notebook candidates ===")
print(stale_notebook_candidates if stale_notebook_candidates else "(none)")

print()
print("[SECTION 01 RESULT]")
print("status: PASS")
print(f"files_found: {len(file_inventory_df)}")
print(f"source_candidates: {len(source_candidates)}")
print(f"conflicting_files: {len(dup_basenames)}")
print(f"stale_notebooks: {len(stale_notebook_candidates)}")
print("next: Registry/PDF integrity audit")


/home/sieg/.local/lib/python3.12/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
WARNING could not parse notebook P3_TARGET/91_24target.ipynb: Notebook does not appear to be JSON: ''


=== git / process context ===
git_branch=P3_MARKED_2020_2024  git_head=b3ff9a01edaf8598eeecb5a5ecfd16bd4dc59bc8
active_ipykernel_processes=19

=== inventory counts ===
pdf=45  notebooks=11  parquet=6  json=4  csv=5  md=23

=== source candidates found ===
(none found -- expect SOURCE_MODE=rebuild_from_pdf in section 03)

=== conflicting (duplicate basename) files ===
                                                           file_path                      basename  file_size
                               P3_BASE/2020_marked_issue_mapping.csv 2020_marked_issue_mapping.csv      67573
P3_TARGET/outputs/marked_parallel/2020/2020_marked_issue_mapping.csv 2020_marked_issue_mapping.csv      67573
                               P3_BASE/2024_marked_issue_mapping.csv 2024_marked_issue_mapping.csv      40554
P3_TARGET/outputs/marked_parallel/2024/2024_marked_issue_mapping.csv 2024_marked_issue_mapping.csv      40554
                                                P3_PROJECT/README.md             

## 02. Registry · PDF 무결성 감사

**목적**: `control_registry.parquet`가 실제 PDF 42개와 완전히 대응하는지 재검증한다(기존 CRAWL_REPORT를 신뢰하지 않고 재확인).
**입력**: `data_parse/pdf_crawler/control_registry.parquet`, `pdf_raw_data/*.pdf`.
**출력**: Registry 통계, PDF 실물 검증(존재/시그니처/open/page_count 일치) 결과.
**가능한 실패**: registry에 있는데 파일 없음(orphan row), 파일은 있는데 registry에 없음(orphan file),
signature 불일치, open 실패, page_count 불일치.


In [4]:
REGISTRY_PATH = PROJECT_ROOT / "data_parse" / "pdf_crawler" / "control_registry.parquet"
assert REGISTRY_PATH.exists(), f"control_registry.parquet not found at {REGISTRY_PATH}"

registry_df = pd.read_parquet(REGISTRY_PATH)

registry_stats = {
    "row_count": len(registry_df),
    "meeting_id_unique": registry_df["meeting_id"].nunique(),
    "meeting_id_missing": int(registry_df["meeting_id"].isna().sum()),
    "meeting_id_duplicated": int(registry_df["meeting_id"].duplicated().sum()),
    "meeting_year_dist": registry_df["meeting_year"].value_counts().sort_index().to_dict(),
    "download_status_dist": registry_df["download_status"].value_counts().to_dict(),
    "parse_status_dist": registry_df["parse_status"].value_counts().to_dict(),
    "page_count_sum": int(registry_df["page_count"].dropna().sum()),
    "local_pdf_path_missing": int(registry_df["local_pdf_path"].isna().sum()),
}
print("=== Registry stats ===")
for k, v in registry_stats.items():
    print(f"{k}: {v}")


=== Registry stats ===
row_count: 42
meeting_id_unique: 42
meeting_id_missing: 0
meeting_id_duplicated: 0
meeting_year_dist: {'20': 7, '21': 7, '22': 7, '23': 7, '24': 7, '25': 7}
download_status_dist: {'downloaded': 42}
parse_status_dist: {'pending': 42}
page_count_sum: 4495
local_pdf_path_missing: 0


In [5]:
PDF_DIR = PROJECT_ROOT / "pdf_raw_data"
actual_pdf_files = sorted(PDF_DIR.glob("*.pdf"))
actual_pdf_stems = {p.stem for p in actual_pdf_files}
registry_stems = set(registry_df["meeting_id"].astype(str))

orphan_registry_rows = registry_stems - actual_pdf_stems   # in registry, no file
orphan_pdf_files = actual_pdf_stems - registry_stems       # file exists, no registry row

pdf_check_rows = []
for p in actual_pdf_files:
    row = {"meeting_id": p.stem, "file_exists": True, "file_size": p.stat().st_size}
    with open(p, "rb") as f:
        head = f.read(8)
    row["signature_ok"] = head.startswith(b"%PDF-")
    try:
        doc = fitz.open(p)
        row["open_ok"] = True
        row["actual_page_count"] = doc.page_count
        doc.close()
    except Exception as e:
        row["open_ok"] = False
        row["actual_page_count"] = None
        row["open_error"] = f"{type(e).__name__}:{e}"
    pdf_check_rows.append(row)

pdf_check_df = pd.DataFrame(pdf_check_rows)
merged = pdf_check_df.merge(
    registry_df[["meeting_id", "page_count", "sha256", "file_size"]].rename(
        columns={"page_count": "registry_page_count", "sha256": "registry_sha256", "file_size": "registry_file_size"}),
    on="meeting_id", how="left",
)
merged["page_count_mismatch"] = merged["actual_page_count"] != merged["registry_page_count"]
merged["sha256_actual"] = [sha256_of_file(PDF_DIR / f"{m}.pdf") for m in merged["meeting_id"]]
merged["sha256_mismatch"] = merged["sha256_actual"] != merged["registry_sha256"]

pdf_integrity_summary = {
    "n_files": len(actual_pdf_files),
    "n_signature_fail": int((~pdf_check_df["signature_ok"]).sum()),
    "n_open_fail": int((~pdf_check_df["open_ok"]).sum()),
    "n_page_count_mismatch": int(merged["page_count_mismatch"].sum()),
    "n_sha256_mismatch": int(merged["sha256_mismatch"].sum()),
    "n_orphan_registry_rows": len(orphan_registry_rows),
    "n_orphan_pdf_files": len(orphan_pdf_files),
}
print("=== PDF integrity ===")
for k, v in pdf_integrity_summary.items():
    print(f"{k}: {v}")
if orphan_registry_rows:
    print("orphan_registry_rows (registry has row, no file):", sorted(orphan_registry_rows))
if orphan_pdf_files:
    print("orphan_pdf_files (file exists, no registry row):", sorted(orphan_pdf_files))
mismatch_sample = merged[merged["page_count_mismatch"] | merged["sha256_mismatch"]]
print()
print("=== mismatch sample (max 10) ===")
print(mismatch_sample.head(10).to_string(index=False) if len(mismatch_sample) else "(none)")

registry_pdf_gate_pass = (
    pdf_integrity_summary["n_signature_fail"] == 0 and pdf_integrity_summary["n_open_fail"] == 0
    and pdf_integrity_summary["n_page_count_mismatch"] == 0 and pdf_integrity_summary["n_sha256_mismatch"] == 0
    and pdf_integrity_summary["n_orphan_registry_rows"] == 0 and pdf_integrity_summary["n_orphan_pdf_files"] == 0
)

print()
print("[SECTION 02 RESULT]")
print("status:", "PASS" if registry_pdf_gate_pass else "FAIL")
print(f"registry_rows: {registry_stats['row_count']}")
print(f"pdf_files: {pdf_integrity_summary['n_files']}")
print(f"failures: sig={pdf_integrity_summary['n_signature_fail']} open={pdf_integrity_summary['n_open_fail']} "
      f"page_count={pdf_integrity_summary['n_page_count_mismatch']} sha256={pdf_integrity_summary['n_sha256_mismatch']} "
      f"orphan_registry={pdf_integrity_summary['n_orphan_registry_rows']} orphan_pdf={pdf_integrity_summary['n_orphan_pdf_files']}")
print("next: Block/Turn source resolution")


=== PDF integrity ===
n_files: 42
n_signature_fail: 0
n_open_fail: 0
n_page_count_mismatch: 0
n_sha256_mismatch: 0
n_orphan_registry_rows: 0
n_orphan_pdf_files: 0

=== mismatch sample (max 10) ===
(none)

[SECTION 02 RESULT]
status: PASS
registry_rows: 42
pdf_files: 42
failures: sig=0 open=0 page_count=0 sha256=0 orphan_registry=0 orphan_pdf=0
next: Block/Turn source resolution


## 03. Block · Turn Source Resolution

**목적**: canonical parquet → pilot cache → PDF 재구축 순으로 block/turn 데이터 출처를 결정한다.
**입력**: 01절에서 만든 `file_inventory_df`의 source 후보, `90_pdf_text_extraction_pilot.ipynb`(파서 계약 확인용, `%run` 안 함).
**출력**: `SOURCE_MODE` 확정값과 근거.
**가능한 실패**: 아무 source도 확보 못 함(`SOURCE_MODE="unavailable"`) — 이 경우 후속 셀은 예외 없이 명시적으로 skip.


In [6]:
block_candidates = file_inventory_df[file_inventory_df["role_candidate"] == "block_source_candidate"]
turn_candidates = file_inventory_df[file_inventory_df["role_candidate"] == "turn_source_candidate"]
page_candidates = file_inventory_df[file_inventory_df["role_candidate"] == "page_source_candidate"]

PILOT_NOTEBOOK_PATH = PROJECT_ROOT / "90_pdf_text_extraction_pilot.ipynb"

if len(block_candidates) and len(turn_candidates):
    SOURCE_MODE = "canonical_parquet" if any("blocks.parquet" in p or "speaker_turns.parquet" in p
                                              for p in list(block_candidates["file_path"]) + list(turn_candidates["file_path"])) \
                  else "pilot_cache"
elif PILOT_NOTEBOOK_PATH.exists() and pdf_integrity_summary["n_files"] > 0:
    SOURCE_MODE = "rebuild_from_pdf"
else:
    SOURCE_MODE = "unavailable"

print(f"block_source_candidates found: {len(block_candidates)}")
print(f"turn_source_candidates found: {len(turn_candidates)}")
print(f"page_source_candidates found: {len(page_candidates)}")
print(f"90_pdf_text_extraction_pilot.ipynb exists: {PILOT_NOTEBOOK_PATH.exists()}")
print()
print(f"SOURCE_MODE = {SOURCE_MODE!r}")

if SOURCE_MODE == "unavailable":
    print("SOURCE_NOT_AVAILABLE")
    print("필요한 파일: blocks.parquet/speaker_turns.parquet 또는 90_pdf_text_extraction_pilot.ipynb + pdf_raw_data/*.pdf")
    print(f"탐색한 경로: {PROJECT_ROOT}")
    print("다음 복구 조치: 02_pdf_crawler.ipynb 재실행으로 pdf_raw_data 확보 필요")


block_source_candidates found: 0
turn_source_candidates found: 0
page_source_candidates found: 0
90_pdf_text_extraction_pilot.ipynb exists: True

SOURCE_MODE = 'rebuild_from_pdf'


### Case C 확인: 90_pdf_text_extraction_pilot.ipynb 파서 계약 검토 (읽기 전용, `%run` 사용 안 함)

`SOURCE_MODE="rebuild_from_pdf"`인 경우, 90 notebook을 JSON으로만 읽어 이미 검증된 파서 계약(컬럼 보정,
정규식 3종, block_no/turn_no ID 규칙)을 확인하고, 아래 04절에서 **이 노트북 안에 독립적으로** 동일 로직을
재정의한다.


In [7]:
if SOURCE_MODE == "rebuild_from_pdf":
    pilot_nb = nbf.read(PILOT_NOTEBOOK_PATH, as_version=4)
    pilot_code = "\n".join(c.source for c in pilot_nb.cells if c.cell_type == "code")
    contract_checks = {
        "adaptive_column_sort (ordered_text_blocks)": "ordered_text_blocks" in pilot_code,
        "PAGE_HEADER_RE defined": "PAGE_HEADER_RE" in pilot_code,
        "TIME_MARKER_RE defined": "TIME_MARKER_RE" in pilot_code,
        "SPEAKER_HEADER_RE defined (^◯)": "SPEAKER_HEADER_RE" in pilot_code,
        "block_no ID format {meeting}_{page:04d}_{seq:04d}": "{page_no:04d}_{block_seq:04d}" in pilot_code,
        "turn_no ID format {meeting}_T{seq:05d}": "_T{turn_seq:05d}" in pilot_code,
    }
    print("=== 90 notebook parser contract check (source inspection only) ===")
    for k, v in contract_checks.items():
        print(f"{k}: {'FOUND' if v else 'MISSING'}")
    assert all(contract_checks.values()), "90 notebook contract drifted -- review before rebuilding"
    print()
    print("Contract confirmed consistent. Section 04 will define an INDEPENDENT reimplementation")
    print("of this exact contract (no %run, no cross-notebook import).")

print()
print("[SECTION 03 RESULT]")
print("status: PASS")
print(f"source_mode: {SOURCE_MODE}")
print("next: DataFrame load or audit rebuild")


=== 90 notebook parser contract check (source inspection only) ===
adaptive_column_sort (ordered_text_blocks): FOUND
PAGE_HEADER_RE defined: FOUND
TIME_MARKER_RE defined: FOUND
SPEAKER_HEADER_RE defined (^◯): FOUND
block_no ID format {meeting}_{page:04d}_{seq:04d}: FOUND
turn_no ID format {meeting}_T{seq:05d}: FOUND

Contract confirmed consistent. Section 04 will define an INDEPENDENT reimplementation
of this exact contract (no %run, no cross-notebook import).

[SECTION 03 RESULT]
status: PASS
source_mode: rebuild_from_pdf
next: DataFrame load or audit rebuild


## 04. DataFrame Load 또는 Audit Rebuild

> **Phase 1 결함 수정 적용됨**: 이전 실행에서 `EMPTY`-type block이 `turn_no` FK는 받으면서
> `n_blocks`/`block_end_no`/`page_end_no` 갱신에서는 빠지는 버그가 있었다(2/65,590 turn 영향,
> §08에서 전수 감사로 발견). `EMPTY`를 `BODY_TEXT`/`IMAGE`와 동일한 공통 경로로 합류시켜 수정했다.
> 이 셀 아래 §08 결과가 `containment_fail=0`, `n_blocks_mismatch=0`으로 나오는지가 이번 재실행의
> 핵심 통과 기준이다.

**목적**: `SOURCE_MODE`에 따라 block_df/turn_df를 확보한다. `rebuild_from_pdf`인 경우 42개 PDF 전체에서
native text를 다시 추출해 **이 노트북 메모리 안에서만** block_df/turn_df를 만든다 — 어떤 canonical parquet도
쓰지 않는다.
**입력**: `pdf_raw_data/*.pdf`, `registry_df`.
**출력**: 메모리 상의 `block_df`, `turn_df` (컬럼: block_no/turn_no PK, 좌표, block_type, block_text/normalized_text,
classification_rule/confidence, turn FK).
**가능한 실패**: 특정 PDF open 실패, 페이지 추출 0건, 컬럼 스키마 불일치.


In [8]:
PAGE_HEADER_RE = re.compile(
    r"^(?:\d+\s{0,4})?\d{4}년도국감-문화체육관광\(\d{4}년\d{1,2}월\d{1,2}일\)(?:\s{0,4}\d+)?$"
)
TIME_MARKER_RE = re.compile(
    r"^\(\d{1,2}시\d{1,2}분\s?[^()]{0,20}(개의|개시|속개|중지|계속|종료|산회|정회|폐회)\)$"
)
SPEAKER_HEADER_RE = re.compile(r"^◯")

def ordered_text_blocks(page):
    """Adaptive per-page column-aware ordering (same contract as 90_pdf_text_extraction_pilot.ipynb)."""
    mid_x = page.rect.width / 2.0
    blocks = [b for b in page.get_text("blocks", sort=True) if b[6] == 0]
    nonempty = [b for b in blocks if b[4].strip()]
    if not nonempty:
        return blocks
    right_ratio = sum(1 for b in nonempty if b[0] > mid_x + 10) / len(nonempty)
    if right_ratio >= 0.15:
        col_of = lambda b: 0 if (b[0] + b[2]) / 2.0 < mid_x else 1
        return sorted(blocks, key=lambda b: (col_of(b), b[1], b[0]))
    return sorted(blocks, key=lambda b: (b[1], b[0]))

def normalized_lines_of_block(raw_text):
    return [ln.strip() for ln in raw_text.split("\n") if ln.strip()]

def classify_block(raw_text):
    lines = normalized_lines_of_block(raw_text)
    if not lines:
        return "EMPTY", "empty_rule"
    first = lines[0]
    if PAGE_HEADER_RE.match(first):
        return "PAGE_HEADER", "PAGE_HEADER_RE"
    if TIME_MARKER_RE.match(first):
        return "TIME_MARKER", "TIME_MARKER_RE"
    if SPEAKER_HEADER_RE.match(first):
        return "SPEAKER_HEADER", "SPEAKER_HEADER_RE"
    return "BODY_TEXT", "fallback_body_text"

def build_block_and_turn_df(meeting_id, pdf_path):
    doc = fitz.open(pdf_path)
    block_rows = []
    for pno in range(len(doc)):
        page = doc[pno]
        for block_seq, b in enumerate(ordered_text_blocks(page)):
            x0, y0, x1, y1, raw_text, bno, btype = b
            block_no = f"{meeting_id}_{pno+1:04d}_{block_seq:04d}"
            norm_lines = normalized_lines_of_block(raw_text)
            if btype == 1:
                block_type, rule = "IMAGE", "image_block"
            else:
                block_type, rule = classify_block(raw_text)
            block_rows.append({
                "block_no": block_no, "meeting_id": meeting_id, "page_no": pno + 1, "block_seq": block_seq,
                "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                "source_block_kind": "IMAGE" if btype == 1 else "TEXT",
                "block_text": raw_text, "normalized_text": "".join(norm_lines),
                "block_type": block_type,
                "classification_rule": rule,
                "classification_confidence": 1.0 if rule != "fallback_body_text" else np.nan,
            })
    doc.close()

    turn_rows, turn_seq, current = [], 0, None
    def open_turn(turn_type, speaker_raw, br):
        nonlocal turn_seq, current
        turn_seq += 1
        current = {
            "turn_no": f"{meeting_id}_T{turn_seq:05d}", "meeting_id": meeting_id, "turn_seq": turn_seq,
            "turn_type": turn_type, "speaker_raw": speaker_raw,
            "page_start_no": br["page_no"], "page_end_no": br["page_no"],
            "block_start_no": br["block_no"], "block_end_no": br["block_no"],
            "block_nos": [], "time_markers": [], "n_speaker_headers": 0,
        }
        turn_rows.append(current)

    for br in block_rows:
        bt = br["block_type"]
        if bt == "PAGE_HEADER":
            br["turn_no"] = None
            continue
        if bt == "TIME_MARKER":
            if current is None:
                open_turn("ORPHAN_FRONT_MATTER", None, br)
            current["time_markers"].append(br["normalized_text"])
        elif bt == "SPEAKER_HEADER":
            open_turn("SPEAKER_TURN", br["normalized_text"], br)
            current["n_speaker_headers"] += 1
        else:  # BODY_TEXT, EMPTY, or IMAGE -- all fall through to the shared bookkeeping below.
            # FIX (Phase 1, was: EMPTY got turn_no set but was `continue`d before being appended to
            # block_nos / advancing block_end_no+page_end_no -- undercounted n_blocks and could leave
            # block_end_no pointing at an earlier block. EMPTY now behaves exactly like BODY_TEXT/IMAGE.
            if current is None:
                open_turn("ORPHAN_FRONT_MATTER", None, br)
        current["block_nos"].append(br["block_no"])
        current["page_end_no"] = br["page_no"]
        current["block_end_no"] = br["block_no"]
        br["turn_no"] = current["turn_no"]

    block_df_one = pd.DataFrame(block_rows)
    norm_by_no = dict(zip(block_df_one["block_no"], block_df_one["normalized_text"]))
    for t in turn_rows:
        t["raw_text"] = "".join(norm_by_no[b] for b in t["block_nos"])
        t["normalized_text"] = t["raw_text"]
        t["char_count"] = len(t["raw_text"])
        t["n_blocks"] = len(t["block_nos"])
        t["sentence_count"] = len(re.findall(r"[.!?]|다\.|까\?|습니다|입니다", t["raw_text"])) or (1 if t["raw_text"] else 0)
        t["is_orphan"] = t["turn_type"] == "ORPHAN_FRONT_MATTER"
    return block_df_one, turn_rows

print("parser functions defined (independent reimplementation, matches 90 notebook contract)")


parser functions defined (independent reimplementation, matches 90 notebook contract)


In [9]:
if SOURCE_MODE != "rebuild_from_pdf":
    raise RuntimeError(f"unexpected SOURCE_MODE={SOURCE_MODE} for this environment -- expected rebuild_from_pdf")

downloaded_rows = registry_df[registry_df["download_status"] == "downloaded"].copy()
print(f"rebuilding block_df/turn_df from {len(downloaded_rows)} downloaded PDFs ...")

_block_dfs, _turn_rows_all = [], []
_rebuild_errors = []
for _, r in downloaded_rows.iterrows():
    mid = r["meeting_id"]
    pdf_path = PDF_DIR / f"{mid}.pdf"
    try:
        b_df, t_rows = build_block_and_turn_df(mid, pdf_path)
        _block_dfs.append(b_df)
        _turn_rows_all.extend(t_rows)
    except Exception as e:
        _rebuild_errors.append({"meeting_id": mid, "error_type": type(e).__name__, "error_message": str(e)})
        logger.error(f"rebuild failed for meeting_id={mid}: {type(e).__name__}: {e}")

block_df = pd.concat(_block_dfs, ignore_index=True) if _block_dfs else pd.DataFrame()
turn_df = pd.DataFrame([{k: v for k, v in t.items() if k != "block_nos"} for t in _turn_rows_all])
_turn_block_nos = {t["turn_no"]: t["block_nos"] for t in _turn_rows_all}

print(f"block_df rows: {len(block_df)}")
print(f"turn_df rows: {len(turn_df)}")
print(f"rebuild errors: {len(_rebuild_errors)}")
for e in _rebuild_errors[:10]:
    print(" ", e)

print()
print("[SECTION 04 RESULT]")
print("status:", "PASS" if not _rebuild_errors else "PARTIAL")
print(f"stage: DATAFRAME_REBUILD (source_mode={SOURCE_MODE})")
print(f"input_pdfs: {len(downloaded_rows)}")
print(f"block_rows: {len(block_df)}")
print(f"turn_rows: {len(turn_df)}")
print(f"failed_meetings: {len(_rebuild_errors)}")
print(f"sample_errors: {_rebuild_errors[:5]}")
print("next: Schema/dtype contract audit")


rebuilding block_df/turn_df from 42 downloaded PDFs ...


block_df rows: 293717
turn_df rows: 65590
rebuild errors: 0

[SECTION 04 RESULT]
status: PASS
stage: DATAFRAME_REBUILD (source_mode=rebuild_from_pdf)
input_pdfs: 42
block_rows: 293717
turn_rows: 65590
failed_meetings: 0
sample_errors: []
next: Schema/dtype contract audit


## 05. Schema · dtype 계약 감사

**목적**: 재구축한 `block_df`/`turn_df`의 실제 컬럼이 SSOT가 요구하는 핵심 컬럼을 전부 포함하는지 확인한다.
**입력**: `block_df`, `turn_df`(메모리), SSOT §6.3/§6.4 핵심 컬럼 목록.
**출력**: expected/actual/missing/extra 컬럼, 계약 상태(MATCH/PARTIAL/MISMATCH).
**가능한 실패**: 핵심 컬럼 누락(MISMATCH), 이름은 있으나 dtype이 기대와 다름(PARTIAL).


In [10]:
EXPECTED_BLOCK_COLUMNS = {
    "block_no": "string", "page_no": "Int64", "meeting_id": "string", "block_seq": "Int64",
    "x0": "Float64", "y0": "Float64", "x1": "Float64", "y1": "Float64",
    "source_block_kind": "category", "block_text": "string", "normalized_text": "string",
    "block_type": "category", "classification_rule": "string", "classification_confidence": "Float64",
    "turn_no": "string",
}
EXPECTED_TURN_COLUMNS = {
    "turn_no": "string", "meeting_id": "string", "turn_seq": "Int64", "turn_type": "category",
    "speaker_raw": "string", "page_start_no": "Int64", "page_end_no": "Int64",
    "block_start_no": "string", "block_end_no": "string", "raw_text": "string",
    "normalized_text": "string", "char_count": "Int64", "sentence_count": "Int64", "is_orphan": "boolean",
}

def schema_contract_report(df, expected_cols, label):
    actual_cols = set(df.columns)
    expected_set = set(expected_cols)
    missing = sorted(expected_set - actual_cols)
    extra = sorted(actual_cols - expected_set)
    status = "MATCH" if not missing else ("PARTIAL" if len(missing) < len(expected_cols) else "MISMATCH")
    print(f"=== {label} schema contract ===")
    print(f"expected: {len(expected_cols)}  actual: {len(actual_cols)}  missing: {missing}  extra: {extra}")
    print(f"status: {status}")
    return {"label": label, "expected": len(expected_cols), "actual": len(actual_cols),
            "missing": missing, "extra": extra, "status": status}

block_schema_report = schema_contract_report(block_df, EXPECTED_BLOCK_COLUMNS, "block_df")
turn_schema_report = schema_contract_report(turn_df, EXPECTED_TURN_COLUMNS, "turn_df")

print()
print("[SECTION 05 RESULT]")
print("status:", "PASS" if block_schema_report["status"] == "MATCH" and turn_schema_report["status"] == "MATCH" else "WARN")
print(f"block_schema: {block_schema_report['status']}  turn_schema: {turn_schema_report['status']}")
print("next: PK/FK/page coverage audit")


=== block_df schema contract ===
expected: 15  actual: 15  missing: []  extra: []
status: MATCH
=== turn_df schema contract ===
expected: 14  actual: 17  missing: []  extra: ['n_blocks', 'n_speaker_headers', 'time_markers']
status: MATCH

[SECTION 05 RESULT]
status: PASS
block_schema: MATCH  turn_schema: MATCH
next: PK/FK/page coverage audit


## 06. PK · FK · 페이지 Coverage 감사

**목적**: `block_no`/`turn_no` PK 유일성, `block.turn_no → turn_df.turn_no` FK 무결성,
Registry의 `page_count` 합과 실제 (meeting_id, page_no) coverage가 일치하는지 확인한다.
**입력**: `block_df`, `turn_df`, `registry_df`.
**출력**: PK 중복 수, FK orphan 수, page coverage 비율, 누락 페이지 목록.
**가능한 실패**: PK 중복(있어서는 안 됨), turn_no FK가 가리키는 대상이 turn_df에 없음, 페이지 누락.


In [11]:
block_pk_dupes = int(block_df["block_no"].duplicated().sum())
turn_pk_dupes = int(turn_df["turn_no"].duplicated().sum())

valid_turn_nos = set(turn_df["turn_no"])
non_null_block_turn = block_df["turn_no"].dropna()
block_fk_orphans = int((~non_null_block_turn.isin(valid_turn_nos)).sum())

valid_meeting_ids = set(registry_df["meeting_id"])
turn_meeting_fk_orphans = int((~turn_df["meeting_id"].isin(valid_meeting_ids)).sum())

# page coverage
actual_pages = block_df.groupby("meeting_id")["page_no"].agg(lambda s: set(s.unique()))
expected_pages_per_meeting = registry_df.set_index("meeting_id")["page_count"]
coverage_rows = []
for mid, pages_seen in actual_pages.items():
    expected_n = int(expected_pages_per_meeting.get(mid, 0))
    expected_set = set(range(1, expected_n + 1))
    missing_pages = sorted(expected_set - pages_seen)
    extra_pages = sorted(pages_seen - expected_set)
    coverage_rows.append({
        "meeting_id": mid, "expected_pages": expected_n, "actual_pages_seen": len(pages_seen),
        "missing_pages": missing_pages, "extra_pages": extra_pages,
    })
coverage_df = pd.DataFrame(coverage_rows)
total_expected_pages = int(registry_df["page_count"].sum())
total_actual_pages = int(coverage_df["actual_pages_seen"].sum())
page_coverage_ratio = total_actual_pages / total_expected_pages if total_expected_pages else None
meetings_with_missing_pages = coverage_df[coverage_df["missing_pages"].map(len) > 0]

print(f"block_no PK duplicates: {block_pk_dupes}")
print(f"turn_no PK duplicates: {turn_pk_dupes}")
print(f"block.turn_no FK orphans (non-null, not in turn_df): {block_fk_orphans}")
print(f"turn.meeting_id FK orphans (not in registry): {turn_meeting_fk_orphans}")
print()
print(f"total_expected_pages (registry sum): {total_expected_pages}")
print(f"total_actual_pages (block_df coverage): {total_actual_pages}")
print(f"page_coverage_ratio: {page_coverage_ratio:.4f}")
print(f"meetings_with_missing_pages: {len(meetings_with_missing_pages)}")
if len(meetings_with_missing_pages):
    print(meetings_with_missing_pages.to_string(index=False))

pk_fk_pass = (block_pk_dupes == 0 and turn_pk_dupes == 0 and block_fk_orphans == 0
              and turn_meeting_fk_orphans == 0 and page_coverage_ratio == 1.0)

print()
print("[SECTION 06 RESULT]")
print("status:", "PASS" if pk_fk_pass else "FAIL")
print(f"block_pk_dupes={block_pk_dupes} turn_pk_dupes={turn_pk_dupes} block_fk_orphans={block_fk_orphans} "
      f"turn_meeting_fk_orphans={turn_meeting_fk_orphans} page_coverage_ratio={page_coverage_ratio}")
print("next: Block physical structure audit")


block_no PK duplicates: 0
turn_no PK duplicates: 0
block.turn_no FK orphans (non-null, not in turn_df): 0
turn.meeting_id FK orphans (not in registry): 0

total_expected_pages (registry sum): 4495
total_actual_pages (block_df coverage): 4495
page_coverage_ratio: 1.0000
meetings_with_missing_pages: 0

[SECTION 06 RESULT]
status: PASS
block_pk_dupes=0 turn_pk_dupes=0 block_fk_orphans=0 turn_meeting_fk_orphans=0 page_coverage_ratio=1.0
next: Block physical structure audit


## 07. Block 물리 구조 감사

**목적**: block 좌표·텍스트 필드의 물리적 유효성과 `block_type` 분포를 확인한다.
**입력**: `block_df`.
**출력**: 좌표 결측/역전 수, 텍스트 결측 수, `source_block_kind`/`block_type` 분포, 분류 신뢰도 분포.
**가능한 실패**: 좌표 결측·역전(x1<x0 또는 y1<y0), block_text 결측, UNKNOWN 비율 과다.


In [12]:
coord_cols = ["x0", "y0", "x1", "y1"]
coord_missing = int(block_df[coord_cols].isna().any(axis=1).sum())
coord_inverted = int(((block_df["x1"] < block_df["x0"]) | (block_df["y1"] < block_df["y0"])).sum())
block_text_missing = int(block_df["block_text"].isna().sum())
normalized_text_missing = int(block_df["normalized_text"].isna().sum())

source_kind_dist = block_df["source_block_kind"].value_counts().to_dict()
block_type_dist = block_df["block_type"].value_counts().to_dict()
n_unknown = int((block_df["block_type"] == "UNKNOWN").sum())
unknown_ratio = n_unknown / len(block_df) if len(block_df) else None

conf_stats = block_df["classification_confidence"].describe().to_dict()
n_conf_nan = int(block_df["classification_confidence"].isna().sum())
n_rule_missing = int(block_df["classification_rule"].isna().sum())

print(f"coordinate missing (any of x0/y0/x1/y1): {coord_missing}")
print(f"coordinate inverted (x1<x0 or y1<y0): {coord_inverted}")
print(f"block_text missing: {block_text_missing}")
print(f"normalized_text missing: {normalized_text_missing}")
print()
print("source_block_kind distribution:", source_kind_dist)
print("block_type distribution:", block_type_dist)
print(f"UNKNOWN count: {n_unknown}  ratio: {unknown_ratio}")
print("  note: this classifier never emits UNKNOWN by design -- non-matching lines fall back to BODY_TEXT")
print("  (validated in 90_pdf_text_extraction_pilot.ipynb: AGENDA_HEADER/INDEX_ENTRY/PAGE_FOOTER are empty")
print("   categories for this corpus, so BODY_TEXT fallback does not silently hide misclassification).")
print()
print(f"classification_confidence NaN count (fallback BODY_TEXT, no deterministic rule): {n_conf_nan} "
      f"({n_conf_nan/len(block_df):.2%})")
print(f"classification_rule missing: {n_rule_missing}")
print(f"classification_confidence describe (non-null only): {conf_stats}")

block_physical_pass = (coord_missing == 0 and coord_inverted == 0 and block_text_missing == 0
                        and normalized_text_missing == 0 and n_rule_missing == 0)

print()
print("[SECTION 07 RESULT]")
print("status:", "PASS" if block_physical_pass else "FAIL")
print(f"coord_missing={coord_missing} coord_inverted={coord_inverted} block_text_missing={block_text_missing} "
      f"unknown_count={n_unknown} unknown_ratio={unknown_ratio}")
print("next: Turn hierarchy exhaustive audit")


coordinate missing (any of x0/y0/x1/y1): 0
coordinate inverted (x1<x0 or y1<y0): 0
block_text missing: 0
normalized_text missing: 0

source_block_kind distribution: {'TEXT': 293717}
block_type distribution: {'BODY_TEXT': 223374, 'SPEAKER_HEADER': 65548, 'PAGE_HEADER': 4495, 'TIME_MARKER': 298, 'EMPTY': 2}
UNKNOWN count: 0  ratio: 0.0
  note: this classifier never emits UNKNOWN by design -- non-matching lines fall back to BODY_TEXT
  (validated in 90_pdf_text_extraction_pilot.ipynb: AGENDA_HEADER/INDEX_ENTRY/PAGE_FOOTER are empty
   categories for this corpus, so BODY_TEXT fallback does not silently hide misclassification).

classification_confidence NaN count (fallback BODY_TEXT, no deterministic rule): 223374 (76.05%)
classification_rule missing: 0
classification_confidence describe (non-null only): {'count': 70343.0, 'mean': 1.0, 'std': 0.0, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 1.0}

[SECTION 07 RESULT]
status: PASS
coord_missing=0 coord_inverted=0 block_text_missin

## 08. Turn 위계구조 전수 감사

**목적**: 표본이 아니라 **65,590개 turn 전체**에 대해 상속(inheritance)·포함(containment)·인접(adjacency)·
원자적 추적(atomicity) 4가지 관계를 전수 검사한다(이전 대화에서 1건+랜덤 5건, 1건+랜덤 2건으로 표본
검증했던 것의 전수화 버전).
**입력**: `block_df`, `turn_df`.
**출력**: `inheritance_fail`, `containment_fail`, `adjacency_fail`, `atomicity_fail`, `n_blocks_mismatch`,
`char_count_mismatch`, `page_range_mismatch`, `multiple_speaker_headers`, `speaker_turn_without_header` — 각 최대 10건 샘플.
**가능한 실패**: 알고리즘 구성상 0건이어야 하는 지표라도 실제로 0이 아니면 버그로 간주한다(가정하지 않고 검증).


In [13]:
block_sorted = block_df.sort_values(["meeting_id", "page_no", "block_seq"]).reset_index(drop=True)

# grouped aggregates over blocks that belong to a turn (turn_no not null)
assigned = block_sorted[block_sorted["turn_no"].notna()]
grp = assigned.groupby("turn_no")
actual_n_blocks = grp.size().rename("actual_n_blocks")
actual_first_block = grp["block_no"].first().rename("actual_block_start_no")
actual_last_block = grp["block_no"].last().rename("actual_block_end_no")
actual_page_min = grp["page_no"].min().rename("actual_page_start_no")
actual_page_max = grp["page_no"].max().rename("actual_page_end_no")
actual_text = grp["normalized_text"].apply(lambda s: "".join(s)).rename("actual_raw_text")
actual_meeting_ids = grp["meeting_id"].apply(lambda s: set(s.unique())).rename("actual_meeting_ids")
actual_n_speaker_headers = grp.apply(lambda d: int((d["block_type"] == "SPEAKER_HEADER").sum()),
                                      include_groups=False).rename("actual_n_speaker_headers")
actual_first_type = grp["block_type"].first().rename("actual_first_block_type")

joined = turn_df.set_index("turn_no").join(
    [actual_n_blocks, actual_first_block, actual_last_block, actual_page_min, actual_page_max,
     actual_text, actual_meeting_ids, actual_n_speaker_headers, actual_first_type]
)

def _inherit_fail(row):
    ids = row["actual_meeting_ids"]
    return not (isinstance(ids, set) and ids == {row["meeting_id"]})

joined["inheritance_fail"] = joined.apply(_inherit_fail, axis=1)
joined["n_blocks_mismatch"] = joined["n_blocks"] != joined["actual_n_blocks"]
joined["char_count_mismatch"] = joined["char_count"] != joined["actual_raw_text"].str.len()
joined["page_range_mismatch"] = (joined["page_start_no"] != joined["actual_page_start_no"]) | \
                                 (joined["page_end_no"] != joined["actual_page_end_no"])
joined["containment_fail"] = (joined["block_start_no"] != joined["actual_block_start_no"]) | \
                              (joined["block_end_no"] != joined["actual_block_end_no"])
joined["multiple_speaker_headers"] = (joined["turn_type"] == "SPEAKER_TURN") & (joined["actual_n_speaker_headers"] > 1)
joined["speaker_turn_without_header"] = (joined["turn_type"] == "SPEAKER_TURN") & \
                                         (joined["actual_first_block_type"] != "SPEAKER_HEADER")

print("Functions/aggregation defined and applied across all", len(joined), "turns.")


Functions/aggregation defined and applied across all 65590 turns.


In [14]:
# adjacency_fail: by construction only PAGE_HEADER blocks may have turn_no null; any OTHER block_type
# with null turn_no is an adjacency violation (an "orphaned" block belonging to no turn).
orphaned_non_header = block_sorted[(block_sorted["turn_no"].isna()) & (block_sorted["block_type"] != "PAGE_HEADER")]
adjacency_fail_count = len(orphaned_non_header)

# atomicity: for EVERY block, the block_no string must self-parse back to its own meeting_id/page_no/block_seq
def _atomic_check(row):
    parts = row["block_no"].rsplit("_", 2)
    derived_mid = "_".join(parts[:-2])
    try:
        return derived_mid == row["meeting_id"] and int(parts[-2]) == row["page_no"] and int(parts[-1]) == row["block_seq"]
    except Exception:
        return False

block_sorted["atomicity_ok"] = block_sorted.apply(_atomic_check, axis=1)
atomicity_fail_count = int((~block_sorted["atomicity_ok"]).sum())

metric_counts = {
    "inheritance_fail": int(joined["inheritance_fail"].sum()),
    "containment_fail": int(joined["containment_fail"].sum()),
    "adjacency_fail": adjacency_fail_count,
    "atomicity_fail": atomicity_fail_count,
    "n_blocks_mismatch": int(joined["n_blocks_mismatch"].sum()),
    "char_count_mismatch": int(joined["char_count_mismatch"].sum()),
    "page_range_mismatch": int(joined["page_range_mismatch"].sum()),
    "multiple_speaker_headers": int(joined["multiple_speaker_headers"].sum()),
    "speaker_turn_without_header": int(joined["speaker_turn_without_header"].sum()),
}
print("=== Turn hierarchy exhaustive audit (all", len(turn_df), "turns,", len(block_df), "blocks) ===")
for k, v in metric_counts.items():
    print(f"{k}: {v}")

print()
print("=== failure samples (max 10 each) ===")
for col in ["inheritance_fail", "containment_fail", "n_blocks_mismatch", "char_count_mismatch",
            "page_range_mismatch", "multiple_speaker_headers", "speaker_turn_without_header"]:
    bad = joined[joined[col]]
    if len(bad):
        print(f"-- {col} ({len(bad)}) --")
        print(bad.head(10)[["meeting_id", "turn_type", "speaker_raw", "char_count", "n_blocks"]].to_string())
if adjacency_fail_count:
    print(f"-- adjacency_fail ({adjacency_fail_count}) --")
    print(orphaned_non_header.head(10)[["meeting_id", "page_no", "block_seq", "block_type", "normalized_text"]].to_string())
if atomicity_fail_count:
    print(f"-- atomicity_fail ({atomicity_fail_count}) --")
    print(block_sorted[~block_sorted["atomicity_ok"]].head(10)[["block_no", "meeting_id", "page_no", "block_seq"]].to_string())

# root-cause diagnosis for any n_blocks_mismatch / containment_fail: check whether an EMPTY-type block
# was assigned this turn_no but not reflected in n_blocks/block_end_no (a known edge case in the build loop:
# EMPTY blocks get turn_no set but are not appended to block_nos / do not advance block_end_no).
mismatched_turns = joined[joined["n_blocks_mismatch"] | joined["containment_fail"]].index.tolist()
if mismatched_turns:
    print()
    print(f"=== root-cause check for {len(mismatched_turns)} mismatched turn(s) ===")
    culprit_blocks = block_sorted[
        (block_sorted["turn_no"].isin(mismatched_turns)) & (block_sorted["block_type"] == "EMPTY")
    ]
    print("EMPTY-type blocks whose turn_no falls inside a mismatched turn (suspected cause):")
    print(culprit_blocks[["block_no", "meeting_id", "turn_no", "page_no", "block_seq", "block_type"]].to_string(index=False))
    print()
    print("DIAGNOSIS: build_block_and_turn_df() sets br[turn_no] for EMPTY-type blocks but does NOT append them")
    print("to current[block_nos] and does NOT advance current[block_end_no]/page_end_no in the EMPTY-type branch.")
    print("This under-counts n_blocks by 1 and can leave block_end_no pointing at the wrong (earlier) block")
    print("whenever an EMPTY block is the last physical block assigned to a turn. Confirmed root cause, not a new bug")
    print("class -- affects exactly 2/65,590 turns (0.003%), matching the 2 EMPTY blocks found corpus-wide in Section 07.")
    logger.warning(f"n_blocks_mismatch/containment_fail root cause confirmed: EMPTY-block accounting gap, "
                    f"{len(mismatched_turns)} turns affected: {mismatched_turns}")

turn_hierarchy_pass = all(v == 0 for v in metric_counts.values())

print()
print("[SECTION 08 RESULT]")
print("status:", "PASS" if turn_hierarchy_pass else "FAIL")
for k, v in metric_counts.items():
    print(f"{k}: {v}")
print("next: Turn distribution/anomaly/text-quality audit")


=== Turn hierarchy exhaustive audit (all 65590 turns, 293717 blocks) ===
inheritance_fail: 0
containment_fail: 0
adjacency_fail: 0
atomicity_fail: 0
n_blocks_mismatch: 0
char_count_mismatch: 0
page_range_mismatch: 0
multiple_speaker_headers: 0
speaker_turn_without_header: 0

=== failure samples (max 10 each) ===

[SECTION 08 RESULT]
status: PASS
inheritance_fail: 0
containment_fail: 0
adjacency_fail: 0
atomicity_fail: 0
n_blocks_mismatch: 0
char_count_mismatch: 0
page_range_mismatch: 0
multiple_speaker_headers: 0
speaker_turn_without_header: 0
next: Turn distribution/anomaly/text-quality audit


## 09. Turn 분포 · 이상치 · 텍스트 품질 감사

**목적**: turn 길이·block 수·page span 분포를 파악하고, 과병합/과분할/공백소실 후보를 식별한다.
**입력**: `turn_df`.
**출력**: 분위수 표, 최장/최단 20건, 과병합/과분할 후보, 공백 소실 의심 turn.
**가능한 실패**: 없음(진단 전용) — 다만 각 후보 리스트가 비어 있는 것은 "문제 없음"이 아니라
"이 휴리스틱으로는 못 잡음"일 수 있어 별도 해석이 필요하다.


In [15]:
HANGUL_RE = re.compile(r"[\uAC00-\uD7A3]")
REPLACEMENT_CHAR = "\ufffd"

def text_quality_metrics(text: str) -> dict:
    if not text:
        return {"whitespace_ratio": None, "avg_word_len": None, "max_nospace_hangul_run": 0,
                "replacement_char_ratio": 0.0, "control_char_count": 0}
    n = len(text)
    ws = sum(1 for c in text if c.isspace())
    words = [w for w in re.split(r"\s+", text) if w]
    avg_word_len = (sum(len(w) for w in words) / len(words)) if words else None
    max_run, cur = 0, 0
    for c in text:
        if HANGUL_RE.match(c):
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 0
    repl_ratio = text.count(REPLACEMENT_CHAR) / n
    control_count = sum(1 for c in text if unicodedata.category(c) == "Cc" and c not in "\n\t")
    return {"whitespace_ratio": ws / n, "avg_word_len": avg_word_len, "max_nospace_hangul_run": max_run,
            "replacement_char_ratio": repl_ratio, "control_char_count": control_count}

print("text_quality_metrics() defined")


text_quality_metrics() defined


In [16]:
turn_speaker_only = turn_df[turn_df["turn_type"] == "SPEAKER_TURN"].copy()
qs = [0, 1, 5, 25, 50, 75, 95, 99, 100]

def quantile_report(series, qs):
    return {f"p{q}": float(np.percentile(series.dropna(), q)) for q in qs}

char_count_q = quantile_report(turn_df["char_count"], qs)
n_blocks_q = quantile_report(turn_df["n_blocks"], qs)
page_span = (turn_df["page_end_no"] - turn_df["page_start_no"] + 1)
page_span_q = quantile_report(page_span, qs)

orphan_ratio = float((turn_df["turn_type"] == "ORPHAN_FRONT_MATTER").mean())
empty_speaker_turn_ratio = float(((turn_df["turn_type"] == "SPEAKER_TURN") & (turn_df["char_count"] <= 0)).mean())

print("turn total:", len(turn_df))
print("meeting count:", turn_df["meeting_id"].nunique())
print("turn_type distribution:", turn_df["turn_type"].value_counts().to_dict())
print("char_count quantiles:", char_count_q)
print("n_blocks quantiles:", n_blocks_q)
print("page_span quantiles:", page_span_q)
print(f"orphan_ratio: {orphan_ratio:.4%}")
print(f"empty_speaker_turn_ratio: {empty_speaker_turn_ratio:.4%}")

p99_char = char_count_q["p99"]
long_turn_mask = (turn_df["char_count"] >= p99_char) | (turn_df["char_count"] >= 5000) | \
                  (turn_df["n_blocks"] >= 200) | (page_span >= 4)
long_turn_candidates = turn_df[long_turn_mask].sort_values("char_count", ascending=False)

short_turn_mask = (turn_df["char_count"] <= 3) | (turn_df["char_count"] <= 10) | (turn_df["n_blocks"] == 1)
short_turn_candidates = turn_df[short_turn_mask].sort_values("char_count")

# over-merge candidates: >=2 speaker headers already checked in section 08 (should be 0); also flag very long
# SPEAKER_TURN with many blocks as a secondary over-merge signal
over_merge_candidates = turn_df[(turn_df["turn_type"] == "SPEAKER_TURN") &
                                 ((turn_df["n_blocks"] >= 150) | (turn_df["char_count"] >= 4000))]

# over-split candidates: same speaker_raw repeated in consecutive turn_seq within a meeting, short turns
turn_sorted = turn_df.sort_values(["meeting_id", "turn_seq"]).reset_index(drop=True)
turn_sorted["prev_speaker"] = turn_sorted.groupby("meeting_id")["speaker_raw"].shift(1)
turn_sorted["next_speaker"] = turn_sorted.groupby("meeting_id")["speaker_raw"].shift(-1)
over_split_candidates = turn_sorted[
    (turn_sorted["speaker_raw"].notna())
    & (turn_sorted["prev_speaker"] == turn_sorted["next_speaker"])
    & (turn_sorted["char_count"] <= 15)
]

# spacing collapse: hangul run >= 20 chars with no space AND avg_word_len very high relative to text
quality_rows = []
for _, row in turn_speaker_only.sample(min(6000, len(turn_speaker_only)), random_state=RANDOM_SEED).iterrows():
    m = text_quality_metrics(row["normalized_text"])
    m["turn_no"] = row["turn_no"]
    quality_rows.append(m)
quality_df = pd.DataFrame(quality_rows).set_index("turn_no")
spacing_collapse_candidates_idx = quality_df[quality_df["max_nospace_hangul_run"] >= 20].index
spacing_collapse_candidates = turn_df[turn_df["turn_no"].isin(spacing_collapse_candidates_idx)]

print()
print(f"long_turn_candidates: {len(long_turn_candidates)}")
print(f"short_turn_candidates: {len(short_turn_candidates)}")
print(f"over_merge_candidates: {len(over_merge_candidates)}")
print(f"over_split_candidates: {len(over_split_candidates)}")
print(f"spacing_collapse_candidates (sampled {len(quality_df)} turns for text-quality scan): {len(spacing_collapse_candidates)}")

display_cols = ["meeting_id", "turn_no", "speaker_raw", "char_count", "n_blocks",
                "page_start_no", "page_end_no"]
print()
print("=== longest 20 turns ===")
print(long_turn_candidates.head(20)[display_cols].to_string(index=False))
print()
print("=== shortest 20 turns ===")
print(short_turn_candidates.head(20)[display_cols].to_string(index=False))
print()
print("=== spacing collapse candidates (up to 20) ===")
print(spacing_collapse_candidates.head(20)[display_cols].to_string(index=False) if len(spacing_collapse_candidates) else "(none in sample)")
print()
print("=== over-merge candidates (up to 20) ===")
print(over_merge_candidates.head(20)[display_cols].to_string(index=False) if len(over_merge_candidates) else "(none)")
print()
print("=== over-split candidates (up to 20) ===")
print(over_split_candidates.head(20)[display_cols].to_string(index=False) if len(over_split_candidates) else "(none)")

print()
print("[SECTION 09 RESULT]")
print("status: PASS (diagnostic section, no hard gate)")
print(f"long_turn_candidates={len(long_turn_candidates)} short_turn_candidates={len(short_turn_candidates)} "
      f"over_merge_candidates={len(over_merge_candidates)} over_split_candidates={len(over_split_candidates)} "
      f"spacing_collapse_candidates={len(spacing_collapse_candidates)}")
print("next: Manual review sample generation")


turn total: 65590
meeting count: 42
turn_type distribution: {'SPEAKER_TURN': 65548, 'ORPHAN_FRONT_MATTER': 42}
char_count quantiles: {'p0': 7.0, 'p1': 10.0, 'p5': 14.0, 'p25': 23.0, 'p50': 41.0, 'p75': 100.0, 'p95': 377.0, 'p99': 913.2200000000012, 'p100': 6729.0}
n_blocks quantiles: {'p0': 1.0, 'p1': 1.0, 'p5': 1.0, 'p25': 1.0, 'p50': 2.0, 'p75': 4.0, 'p95': 16.0, 'p99': 38.0, 'p100': 333.0}
page_span quantiles: {'p0': 1.0, 'p1': 1.0, 'p5': 1.0, 'p25': 1.0, 'p50': 1.0, 'p75': 1.0, 'p95': 1.0, 'p99': 2.0, 'p100': 6.0}
orphan_ratio: 0.0640%
empty_speaker_turn_ratio: 0.0000%



long_turn_candidates: 656
short_turn_candidates: 24482
over_merge_candidates: 13
over_split_candidates: 102
spacing_collapse_candidates (sampled 6000 turns for text-quality scan): 1703

=== longest 20 turns ===
meeting_id        turn_no                           speaker_raw  char_count  n_blocks  page_start_no  page_end_no
    051381  051381_T00005              ◯문화재청장 김현모  존경하는 문화체육관광위        6729       333              2            6
    050430  050430_T00003              ◯문화체육관광부장관 박양우  존경하는 도종환        5742       276              2            5
    052280  052280_T00005              ◯문화재청장 최응천  존경하는 문화체육관광위        5601       267              2            5
    053680  053680_T00005              ◯문화재청장 최응천  존경하는 문화체육관광위        5558       266              2            5
    050350  050350_T00005              ◯문화재청장 정재숙  존경하는 문화체육관광위        5177       250              2            5
    052229  052229_T00034             ◯문화체육관광부장관 박보균  존경하는 국회 문        4732       236              5    

## 10. 수작업 검토 표본 생성

**목적**: 여러 층(초기/중간/최신 연도, 길이 극단, orphan, procedural, spacing collapse 등)에서 중복 없이
표본을 뽑아 사람이 직접 라벨링할 수 있는 CSV를 만든다.
**입력**: `turn_df`, 09절에서 계산한 후보 목록.
**출력**: `turn_manual_review_sample.csv` (최소 50행), 층별 3~5행 미리보기.
**가능한 실패**: 특정 층에 표본이 부족(예: orphan turn이 42개뿐이라 그 안에서만 뽑음).


In [17]:
def sample_group(df, name, n, exclude_idx):
    pool = df[~df.index.isin(exclude_idx)]
    n = min(n, len(pool))
    if n == 0:
        return pd.DataFrame()
    picked = pool.sample(n, random_state=RANDOM_SEED)
    picked = picked.copy()
    picked["review_group"] = name
    return picked

used_idx = set()
groups = []

turn_df_idx = turn_df.set_index(turn_df.index)
year_of = registry_df.set_index("meeting_id")["meeting_year"]
turn_df["meeting_year"] = turn_df["meeting_id"].map(year_of)

for label, year in [("year_2020", "20"), ("year_mid_2022", "22"), ("year_latest_2025", "25")]:
    pool = turn_df[(turn_df["meeting_year"] == year) & (turn_df["turn_type"] == "SPEAKER_TURN")]
    g = sample_group(pool, label, 6, used_idx)
    used_idx |= set(g.index); groups.append(g)

median_len = char_count_q["p50"]
g = sample_group(turn_df[(turn_df["char_count"].between(median_len*0.9, median_len*1.1))], "median_length", 6, used_idx)
used_idx |= set(g.index); groups.append(g)

g = sample_group(long_turn_candidates, "p95_plus", 6, used_idx); used_idx |= set(g.index); groups.append(g)
g = sample_group(turn_df[turn_df["char_count"] >= char_count_q["p99"]], "p99_plus", 6, used_idx)
used_idx |= set(g.index); groups.append(g)
g = sample_group(short_turn_candidates, "very_short", 6, used_idx); used_idx |= set(g.index); groups.append(g)
g = sample_group(spacing_collapse_candidates, "spacing_collapse_suspect", 5, used_idx)
used_idx |= set(g.index); groups.append(g)
g = sample_group(turn_df[turn_df["turn_type"] == "ORPHAN_FRONT_MATTER"], "orphan", 5, used_idx)
used_idx |= set(g.index); groups.append(g)
g = sample_group(turn_df[turn_df["time_markers"].map(len) > 0], "procedural_time_marker", 5, used_idx)
used_idx |= set(g.index); groups.append(g)
g = sample_group(over_split_candidates, "speaker_boundary_suspect", 5, used_idx)
used_idx |= set(g.index); groups.append(g)
g = sample_group(turn_df[turn_df["turn_type"] == "SPEAKER_TURN"], "random_normal", 8, used_idx)
used_idx |= set(g.index); groups.append(g)

review_sample = pd.concat([g for g in groups if len(g)], ignore_index=True)

def _split_speaker(raw):
    if pd.isna(raw) or not isinstance(raw, str) or not raw:
        return None, None
    m = re.match(r"^◯(.+?)\s{2,}", raw)
    core = m.group(1) if m else raw[1:]
    role_m = re.search(r"(위원장|위원|장관|차관|처장|청장|국장|과장|실장|대표|증인|참고인|본부장)$", core)
    return core, (role_m.group(1) if role_m else None)

_split_result = review_sample["speaker_raw"].map(_split_speaker)
review_sample["speaker_name"] = _split_result.map(lambda t: t[0])
review_sample["speaker_role"] = _split_result.map(lambda t: t[1])
review_sample["suspected_issue"] = review_sample["review_group"]
review_sample["manual_label"] = ""
review_sample["review_note"] = ""

REVIEW_COLUMNS = ["review_group", "meeting_id", "turn_no", "turn_type", "speaker_raw", "speaker_name",
                   "speaker_role", "page_start_no", "page_end_no", "block_start_no", "block_end_no",
                   "char_count", "n_blocks", "normalized_text", "suspected_issue", "manual_label", "review_note"]
review_sample_out = review_sample[REVIEW_COLUMNS]

REVIEW_CSV_PATH = OUTPUT_ROOT / "turn_manual_review_sample.csv"
review_sample_out.to_csv(REVIEW_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"review sample rows: {len(review_sample_out)}")
print(review_sample_out["review_group"].value_counts().to_dict())
print(f"saved: {REVIEW_CSV_PATH}")

print()
print("=== 3~5 rows per group (preview) ===")
for g_name, g_df in review_sample_out.groupby("review_group"):
    print(f"-- {g_name} ({len(g_df)}) --")
    print(g_df.head(4)[["meeting_id", "turn_no", "speaker_raw", "char_count"]].to_string(index=False))

print()
print("[SECTION 10 RESULT]")
print("status:", "PASS" if len(review_sample_out) >= 50 else "WARN")
print(f"review_sample_rows: {len(review_sample_out)}")
print(f"groups: {review_sample_out['review_group'].nunique()}")
print("next: Retrieval Segment Builder")


review sample rows: 70
{'random_normal': 8, 'year_2020': 6, 'year_mid_2022': 6, 'year_latest_2025': 6, 'median_length': 6, 'p95_plus': 6, 'p99_plus': 6, 'very_short': 6, 'spacing_collapse_suspect': 5, 'orphan': 5, 'procedural_time_marker': 5, 'speaker_boundary_suspect': 5}
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_validation_audit/turn_manual_review_sample.csv

=== 3~5 rows per group (preview) ===
-- median_length (6) --
meeting_id        turn_no                         speaker_raw  char_count
   N053481 N053481_T00621           ◯박정하위원청장님, 첫국감인데고생많이하십니다.          41
    050350  050350_T01416           ◯위원장 도종환  서면질의라는 좋은 제도도 있          45
    053357  053357_T00589         ◯이용 위원  관리를 해야 되는 게 마땅한 것 아          40
   N053487 N053487_T01699 ◯국립현대미술관장김성희제가그냥생각하는업체인것같은데요정확히는모르겠          39
-- orphan (5) --
meeting_id       turn_no speaker_raw  char_count
    051381 051381_T00001         NaN         145
    054622 054622_T00001         NaN         105
    

## 11. Retrieval Segment Builder

**목적**: `turn_df`를 `meeting_id`·`turn_seq` 순으로 정렬해 SEG_TURN/SEG_PREV_CURR/SEG_CURR_NEXT 3종을
**audit 전용 메모리 DataFrame**으로 생성한다. canonical `retrieval_segments.parquet`는 만들지 않는다.
**입력**: `turn_df`.
**출력**: `segment_df`(메모리).
**가능한 실패**: 경계 turn(첫/마지막) 처리 누락, procedural/orphan turn의 flag 누락.


In [18]:
turn_sorted2 = turn_df.sort_values(["meeting_id", "turn_seq"]).reset_index(drop=True)
turn_by_no = turn_sorted2.set_index("turn_no")

def _speaker_of(turn_no):
    if turn_no is None:
        return None
    return turn_by_no.loc[turn_no, "speaker_raw"]

def _seg_hash(meeting_id, segment_type, turn_nos, normalized_text):
    payload = f"{meeting_id}|{segment_type}|{'|'.join(turn_nos)}|{normalized_text}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

segment_rows = []
for meeting_id, grp in turn_sorted2.groupby("meeting_id"):
    grp = grp.reset_index(drop=True)
    n = len(grp)
    for i in range(n):
        row = grp.iloc[i]
        anchor_no = row["turn_no"]
        anchor_seq = int(row["turn_seq"])
        prev_row = grp.iloc[i - 1] if i > 0 else None
        next_row = grp.iloc[i + 1] if i < n - 1 else None
        is_anchor_procedural_or_orphan = row["is_orphan"] or bool(row["time_markers"])

        # SEG_TURN -- always present
        seg_turn_text = row["normalized_text"]
        segment_rows.append({
            "segment_no": f"{meeting_id}_SEG_TURN_{anchor_seq:05d}", "meeting_id": meeting_id,
            "segment_type": "SEG_TURN", "anchor_turn_no": anchor_no,
            "turn_start_no": anchor_no, "turn_end_no": anchor_no,
            "prev_turn_no": None, "next_turn_no": None,
            "page_start_no": int(row["page_start_no"]), "page_end_no": int(row["page_end_no"]),
            "speaker_names": row["speaker_raw"], "speaker_roles": None,
            "raw_text": seg_turn_text, "normalized_text": seg_turn_text,
            "char_count": len(seg_turn_text), "sentence_count": int(row["sentence_count"]),
            "anchor_turn_type": row["turn_type"], "anchor_is_procedural_or_orphan": is_anchor_procedural_or_orphan,
        })

        if RETRIEVAL_BOUNDARY_POLICY == "omit_missing_neighbor":
            if prev_row is not None:
                text = prev_row["normalized_text"] + row["normalized_text"]
                segment_rows.append({
                    "segment_no": f"{meeting_id}_SEG_PREV_CURR_{anchor_seq:05d}", "meeting_id": meeting_id,
                    "segment_type": "SEG_PREV_CURR", "anchor_turn_no": anchor_no,
                    "turn_start_no": prev_row["turn_no"], "turn_end_no": anchor_no,
                    "prev_turn_no": prev_row["turn_no"], "next_turn_no": None,
                    "page_start_no": int(prev_row["page_start_no"]), "page_end_no": int(row["page_end_no"]),
                    "speaker_names": f"{prev_row['speaker_raw']} | {row['speaker_raw']}",
                    "speaker_roles": None, "raw_text": text, "normalized_text": text,
                    "char_count": len(text), "sentence_count": int(prev_row["sentence_count"]) + int(row["sentence_count"]),
                    "anchor_turn_type": row["turn_type"], "anchor_is_procedural_or_orphan": is_anchor_procedural_or_orphan,
                })
            if next_row is not None:
                text = row["normalized_text"] + next_row["normalized_text"]
                segment_rows.append({
                    "segment_no": f"{meeting_id}_SEG_CURR_NEXT_{anchor_seq:05d}", "meeting_id": meeting_id,
                    "segment_type": "SEG_CURR_NEXT", "anchor_turn_no": anchor_no,
                    "turn_start_no": anchor_no, "turn_end_no": next_row["turn_no"],
                    "prev_turn_no": None, "next_turn_no": next_row["turn_no"],
                    "page_start_no": int(row["page_start_no"]), "page_end_no": int(next_row["page_end_no"]),
                    "speaker_names": f"{row['speaker_raw']} | {next_row['speaker_raw']}",
                    "speaker_roles": None, "raw_text": text, "normalized_text": text,
                    "char_count": len(text), "sentence_count": int(row["sentence_count"]) + int(next_row["sentence_count"]),
                    "anchor_turn_type": row["turn_type"], "anchor_is_procedural_or_orphan": is_anchor_procedural_or_orphan,
                })
        else:
            raise NotImplementedError(f"RETRIEVAL_BOUNDARY_POLICY={RETRIEVAL_BOUNDARY_POLICY!r} not implemented")

segment_df = pd.DataFrame(segment_rows)
segment_df["segment_hash"] = segment_df.apply(
    lambda r: _seg_hash(r["meeting_id"], r["segment_type"], [r["turn_start_no"], r["turn_end_no"]], r["normalized_text"]),
    axis=1,
)
segment_df["tfidf_ready"] = segment_df["char_count"] > 0

print(f"segment_df rows: {len(segment_df)}")
print(segment_df["segment_type"].value_counts().to_dict())
print(f"procedural/orphan-anchored segments: {int(segment_df['anchor_is_procedural_or_orphan'].sum())}")

print()
print("[SECTION 11 RESULT]")
print("status: PASS")
print(f"segment_rows: {len(segment_df)}")
print(f"segment_type_dist: {segment_df['segment_type'].value_counts().to_dict()}")
print("next: Retrieval Segment 무결성 감사")


segment_df rows: 196686
{'SEG_TURN': 65590, 'SEG_CURR_NEXT': 65548, 'SEG_PREV_CURR': 65548}
procedural/orphan-anchored segments: 528

[SECTION 11 RESULT]
status: PASS
segment_rows: 196686
segment_type_dist: {'SEG_TURN': 65590, 'SEG_CURR_NEXT': 65548, 'SEG_PREV_CURR': 65548}
next: Retrieval Segment 무결성 감사


## 12. Retrieval Segment 무결성 감사

**목적**: segment PK 유일성, turn FK 무결성, 기대 segment 수(`3n-2`) 대비 실제 수, 원문 조립·역추적률을 검증한다.
**입력**: `segment_df`, `turn_df`.
**출력**: segment 총수/분포, expected vs observed, traceability_rate, 이상 segment 샘플.
**가능한 실패**: FK orphan, 조립 텍스트 불일치, 기대 수와 실제 수 불일치(procedural/orphan turn 처리 방식 차이로 발생 가능).


In [19]:
segment_no_dupes = int(segment_df["segment_no"].duplicated().sum())
valid_turn_set = set(turn_df["turn_no"])

def _fk_orphan_count(col):
    vals = segment_df[col].dropna()
    return int((~vals.isin(valid_turn_set)).sum())

fk_orphans = {c: _fk_orphan_count(c) for c in ["anchor_turn_no", "turn_start_no", "turn_end_no", "prev_turn_no", "next_turn_no"]}

meeting_mismatch = int((segment_df["meeting_id"].map(lambda m: m not in set(registry_df["meeting_id"]))).sum())
normalized_text_missing = int(segment_df["normalized_text"].isna().sum())
segment_hash_missing = int(segment_df["segment_hash"].isna().sum())
segment_hash_dupes = int(segment_df["segment_hash"].duplicated().sum())
allowed_types = {"SEG_TURN", "SEG_PREV_CURR", "SEG_CURR_NEXT"}
segment_type_invalid = int((~segment_df["segment_type"].isin(allowed_types)).sum())
dup_anchor_type = int(segment_df.duplicated(["anchor_turn_no", "segment_type"]).sum())

anchor_seg_counts = segment_df.groupby("anchor_turn_no")["segment_type"].nunique()
anchors_with_all_3 = int((anchor_seg_counts == 3).sum())

print("segment_no duplicates:", segment_no_dupes)
print("FK orphans:", fk_orphans)
print("meeting_id mismatch (not in registry):", meeting_mismatch)
print("normalized_text missing:", normalized_text_missing)
print("segment_hash missing:", segment_hash_missing)
print(f"segment_hash duplicates: {segment_hash_dupes} "
      f"(expected: many short identical exchanges like \'예.\' legitimately hash the same -- not automatically a bug)")
print("segment_type invalid:", segment_type_invalid)
print("duplicate (anchor_turn_no, segment_type) pairs:", dup_anchor_type)
print("anchors with all 3 segment types present:", anchors_with_all_3, "/", segment_df["anchor_turn_no"].nunique())


segment_no duplicates: 0
FK orphans: {'anchor_turn_no': 0, 'turn_start_no': 0, 'turn_end_no': 0, 'prev_turn_no': 0, 'next_turn_no': 0}
meeting_id mismatch (not in registry): 0
normalized_text missing: 0
segment_hash missing: 0
segment_hash duplicates: 0 (expected: many short identical exchanges like '예.' legitimately hash the same -- not automatically a bug)
segment_type invalid: 0
duplicate (anchor_turn_no, segment_type) pairs: 0
anchors with all 3 segment types present: 65506 / 65590


In [20]:
# expected vs observed segment count per meeting: n==1 -> 1, n>=2 -> 3n-2
n_per_meeting = turn_df.groupby("meeting_id").size()
expected_per_meeting = n_per_meeting.map(lambda n: 1 if n == 1 else 3 * n - 2)
observed_per_meeting = segment_df.groupby("meeting_id").size()
expected_vs_observed = pd.DataFrame({"n_turns": n_per_meeting, "expected_segments": expected_per_meeting,
                                     "observed_segments": observed_per_meeting}).fillna(0).astype(int)
expected_vs_observed["match"] = expected_vs_observed["expected_segments"] == expected_vs_observed["observed_segments"]
total_expected = int(expected_vs_observed["expected_segments"].sum())
total_observed = int(expected_vs_observed["observed_segments"].sum())
n_meetings_mismatch = int((~expected_vs_observed["match"]).sum())

print("=== expected vs observed segment count per meeting ===")
print(expected_vs_observed.to_string())
print(f"TOTAL expected={total_expected}  observed={total_observed}  meetings_mismatched={n_meetings_mismatch}")


=== expected vs observed segment count per meeting ===
            n_turns  expected_segments  observed_segments  match
meeting_id                                                      
050308         1069               3205               3205   True
050350         1447               4339               4339   True
050430         1240               3718               3718   True
050492         1620               4858               4858   True
050606         1695               5083               5083   True
050615         1534               4600               4600   True
050673         1198               3592               3592   True
051354         1203               3607               3607   True
051381         1127               3379               3379   True
051402         1166               3496               3496   True
051424         1271               3811               3811   True
051487         1224               3670               3670   True
051535         1368               4

In [21]:
# text assembly + traceability verification
turn_text_by_no = turn_df.set_index("turn_no")["normalized_text"]

def _assembly_ok(row):
    try:
        if row["segment_type"] == "SEG_TURN":
            expected = turn_text_by_no[row["anchor_turn_no"]]
        elif row["segment_type"] == "SEG_PREV_CURR":
            expected = turn_text_by_no[row["prev_turn_no"]] + turn_text_by_no[row["anchor_turn_no"]]
        else:  # SEG_CURR_NEXT
            expected = turn_text_by_no[row["anchor_turn_no"]] + turn_text_by_no[row["next_turn_no"]]
        return expected == row["normalized_text"]
    except KeyError:
        return False

segment_df["assembly_ok"] = segment_df.apply(_assembly_ok, axis=1)
assembly_fail_count = int((~segment_df["assembly_ok"]).sum())

# traceability: every referenced turn (anchor/prev/next when present) must exist in turn_df,
# AND that turn must itself be free of the section-08 hierarchy failures (containment/n_blocks mismatch)
_bad_turns = set(joined[joined["containment_fail"] | joined["n_blocks_mismatch"]].index)
def _traceable(row):
    refs = [row["anchor_turn_no"], row["turn_start_no"], row["turn_end_no"]]
    if pd.notna(row["prev_turn_no"]): refs.append(row["prev_turn_no"])
    if pd.notna(row["next_turn_no"]): refs.append(row["next_turn_no"])
    if any(r not in valid_turn_set for r in refs):
        return False
    if any(r in _bad_turns for r in refs):
        return False
    return True

segment_df["traceable"] = segment_df.apply(_traceable, axis=1)
traceability_rate = float(segment_df["traceable"].mean())

print(f"assembly_fail_count: {assembly_fail_count} / {len(segment_df)}")
print(f"traceability_rate: {traceability_rate:.6f}  (target 1.00; known gap = 2 turns from Section 08 EMPTY-block issue)")

invalid_sample = segment_df[~segment_df["assembly_ok"]]
print()
print("=== invalid segment sample (assembly mismatch, max 10) ===")
print(invalid_sample.head(10)[["segment_no","meeting_id","segment_type","anchor_turn_no","char_count"]].to_string(index=False)
      if len(invalid_sample) else "(none)")

segment_integrity_pass = (segment_no_dupes == 0 and all(v == 0 for v in fk_orphans.values())
                           and meeting_mismatch == 0 and segment_hash_missing == 0
                           and segment_type_invalid == 0 and dup_anchor_type == 0 and assembly_fail_count == 0)

print()
print("[SECTION 12 RESULT]")
print("status:", "PASS" if segment_integrity_pass else "WARN")
print(f"segment_total={len(segment_df)} expected_total={total_expected} observed_total={total_observed} "
      f"fk_orphans={sum(fk_orphans.values())} assembly_fail={assembly_fail_count} traceability_rate={traceability_rate:.6f}")
print("next: Long segment / chunk readiness diagnostics")


assembly_fail_count: 0 / 196686
traceability_rate: 1.000000  (target 1.00; known gap = 2 turns from Section 08 EMPTY-block issue)

=== invalid segment sample (assembly mismatch, max 10) ===
(none)

[SECTION 12 RESULT]
status: PASS
segment_total=196686 expected_total=196686 observed_total=196686 fk_orphans=0 assembly_fail=0 traceability_rate=1.000000
next: Long segment / chunk readiness diagnostics


## 13. 긴 Segment · 검색 Chunk 준비도

**목적**: canonical segment는 자르지 않되, 청킹이 필요할 후보와 예상 chunk 수를 진단만 한다.
**입력**: `segment_df`.
**출력**: char_count 분위수, 2,000/5,000/10,000자 초과 수, meeting별 예상 chunk 수, 공백 소실 문서의 chunk 위험.
**가능한 실패**: 없음(진단 전용, canonical export 없음).


In [22]:
seg_char_q = quantile_report(segment_df["char_count"], [0,25,50,75,90,95,99,100])
n_over_2000 = int((segment_df["char_count"] > 2000).sum())
n_over_5000 = int((segment_df["char_count"] > 5000).sum())
n_over_10000 = int((segment_df["char_count"] > 10000).sum())
n_too_short = int((segment_df["char_count"] < 5).sum())
dup_text_count = int(segment_df["normalized_text"].duplicated().sum())

def _est_chunks(n_chars):
    if n_chars <= TARGET_CHUNK_MAX:
        return 1
    step = TARGET_CHUNK_MAX - CHUNK_OVERLAP
    return int(np.ceil((n_chars - CHUNK_OVERLAP) / step))

needs_chunk = segment_df[segment_df["char_count"] > TARGET_CHUNK_MAX].copy()
needs_chunk["est_chunks"] = needs_chunk["char_count"].map(_est_chunks)
total_est_chunks = int(needs_chunk["est_chunks"].sum()) + int((segment_df["char_count"] <= TARGET_CHUNK_MAX).sum())
per_meeting_chunks = needs_chunk.groupby("meeting_id")["est_chunks"].sum()

spacing_risk_meetings = set(spacing_collapse_candidates["meeting_id"].unique())
chunk_risk_from_spacing = needs_chunk[needs_chunk["meeting_id"].isin(spacing_risk_meetings)]

print("segment char_count quantiles:", seg_char_q)
print(f">2000 chars: {n_over_2000}  >5000 chars: {n_over_5000}  >10000 chars: {n_over_10000}  <5 chars: {n_too_short}")
print(f"duplicate normalized_text segments: {dup_text_count}")
print()
print(f"segments needing chunking (> {TARGET_CHUNK_MAX}): {len(needs_chunk)}")
print(f"estimated total chunk count if chunked now: {total_est_chunks}")
print(f"chunk-risk segments in spacing-collapse-suspect meetings: {len(chunk_risk_from_spacing)}")
print()
print("=== longest 10 segments ===")
print(segment_df.sort_values("char_count", ascending=False).head(10)[
    ["segment_no","meeting_id","segment_type","char_count"]].to_string(index=False))

print()
print("[SECTION 13 RESULT]")
print("status: PASS (diagnostic only, no canonical chunk export)")
print(f"needs_chunk_segments={len(needs_chunk)} estimated_total_chunks={total_est_chunks}")
print("next: Target CSV compatibility audit")


segment char_count quantiles: {'p0': 7.0, 'p25': 46.0, 'p50': 89.0, 'p75': 191.0, 'p90': 382.0, 'p95': 564.0, 'p99': 1239.0, 'p100': 7584.0}
>2000 chars: 490  >5000 chars: 27  >10000 chars: 0  <5 chars: 0
duplicate normalized_text segments: 75514

segments needing chunking (> 1500): 1271
estimated total chunk count if chunked now: 198158
chunk-risk segments in spacing-collapse-suspect meetings: 1235

=== longest 10 segments ===


                segment_no meeting_id  segment_type  char_count
050430_SEG_CURR_NEXT_00002     050430 SEG_CURR_NEXT        7584
050430_SEG_PREV_CURR_00003     050430 SEG_PREV_CURR        7584
051381_SEG_PREV_CURR_00005     051381 SEG_PREV_CURR        6955
051381_SEG_CURR_NEXT_00004     051381 SEG_CURR_NEXT        6955
051381_SEG_PREV_CURR_00006     051381 SEG_PREV_CURR        6800
051381_SEG_CURR_NEXT_00005     051381 SEG_CURR_NEXT        6800
     051381_SEG_TURN_00005     051381      SEG_TURN        6729
052280_SEG_PREV_CURR_00005     052280 SEG_PREV_CURR        5821
052280_SEG_CURR_NEXT_00004     052280 SEG_CURR_NEXT        5821
050430_SEG_CURR_NEXT_00003     050430 SEG_CURR_NEXT        5801

[SECTION 13 RESULT]
status: PASS (diagnostic only, no canonical chunk export)
needs_chunk_segments=1271 estimated_total_chunks=198158
next: Target CSV compatibility audit


## 14. Target CSV 호환성 감사

**목적**: 2020·2024 `marked_issue_mapping.csv`(P3_TARGET, 수정하지 않고 읽기만 함)가 이번 corpus와
연도로 연결될 수 있는지 확인하고, TF-IDF 검색에 쓸 query 필드를 파생한다.
**입력**: `P3_TARGET/outputs/marked_parallel/{2020,2024}/*_marked_issue_mapping.csv`.
**출력**: target 행 수, `target_issue_id` 중복, literal `"null"` 처리, 연도별 corpus 호환성.
**가능한 실패**: literal `"null"`을 실제 결측으로 오인, target 연도가 corpus에 없음.


In [23]:
TARGET_CSV_PATHS = {
    "2020": PROJECT_ROOT / "P3_TARGET" / "outputs" / "marked_parallel" / "2020" / "2020_marked_issue_mapping.csv",
    "2024": PROJECT_ROOT / "P3_TARGET" / "outputs" / "marked_parallel" / "2024" / "2024_marked_issue_mapping.csv",
}
for y, p in TARGET_CSV_PATHS.items():
    print(f"{y}: {p}  exists={p.exists()}")

target_dfs = {}
for year, path in TARGET_CSV_PATHS.items():
    df = pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, na_filter=False)
    df["meeting_year_2digit"] = year[2:]
    df["target_issue_id"] = df.apply(lambda r, y=year: f"TGT_{y}_{int(r['issue_no']):04d}", axis=1)
    target_dfs[year] = df
    print(f"{year}: rows={len(df)} columns={list(df.columns)}")


2020: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/outputs/marked_parallel/2020/2020_marked_issue_mapping.csv  exists=True
2024: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/outputs/marked_parallel/2024/2024_marked_issue_mapping.csv  exists=True
2020: rows=110 columns=['meeting_id', 'meeting_year', 'issue_no', 'issue_text', 'action_text', 'future_plan_text', 'status', 'source_page', 'meeting_year_2digit', 'target_issue_id']
2024: rows=71 columns=['meeting_id', 'meeting_year', 'issue_no', 'issue_text', 'action_text', 'future_plan_text', 'status', 'source_page', 'meeting_year_2digit', 'target_issue_id']


In [24]:
def _query_fields(row):
    issue_text = row["issue_text"]
    action_text = row["action_text"]
    future_text = row["future_plan_text"]
    query_core = issue_text
    query_expanded = issue_text + (action_text if action_text != "null" else "")
    parts_full = [issue_text]
    if action_text != "null":
        parts_full.append(action_text)
    if future_text != "null":
        parts_full.append(future_text)
    query_full = "".join(parts_full)
    return pd.Series({"query_core": query_core, "query_expanded": query_expanded, "query_full": query_full})

corpus_years = set(registry_df["meeting_year"])
target_compat_summary = {}
for year, df in target_dfs.items():
    df[["query_core", "query_expanded", "query_full"]] = df.apply(_query_fields, axis=1)
    dup_ids = int(df["target_issue_id"].duplicated().sum())
    issue_text_missing = int((df["issue_text"].str.strip() == "").sum())
    action_null_literal = int((df["action_text"] == "null").sum())
    future_null_literal = int((df["future_plan_text"] == "null").sum())
    status_dist = df["status"].value_counts().to_dict()
    year_2digit = year[2:]
    year_in_corpus = year_2digit in corpus_years
    candidate_meetings = registry_df[registry_df["meeting_year"] == year_2digit]["meeting_id"].tolist()
    candidate_meeting_count = len(candidate_meetings)
    candidate_segment_count = int((segment_df["meeting_id"].isin(candidate_meetings)).sum())
    target_compat_summary[year] = {
        "rows": len(df), "target_issue_id_dupes": dup_ids, "issue_text_missing": issue_text_missing,
        "action_null_literal": action_null_literal, "future_null_literal": future_null_literal,
        "status_dist": status_dist, "year_in_corpus": year_in_corpus,
        "candidate_meeting_count": candidate_meeting_count, "candidate_segment_count": candidate_segment_count,
    }
    print(f"=== {year} ===")
    for k, v in target_compat_summary[year].items():
        print(f"  {k}: {v}")

print()
print("=== sample rows (target_issue_id / query_core / query_expanded / query_full trimmed) ===")
for year, df in target_dfs.items():
    print(f"-- {year} (first 3) --")
    preview = df[["target_issue_id","query_core","query_expanded","query_full"]].head(3).copy()
    for c in ["query_core","query_expanded","query_full"]:
        preview[c] = preview[c].str.slice(0, 60)
    print(preview.to_string(index=False))

target_compat_pass = all(s["target_issue_id_dupes"] == 0 and s["year_in_corpus"] and s["candidate_meeting_count"] > 0
                           for s in target_compat_summary.values())

print()
print("[SECTION 14 RESULT]")
print("status:", "PASS" if target_compat_pass else "WARN")
print(f"target_rows_2020={target_compat_summary['2020']['rows']} target_rows_2024={target_compat_summary['2024']['rows']}")
print("next: Retrieval smoke test")


=== 2020 ===
  rows: 110
  target_issue_id_dupes: 0
  issue_text_missing: 0
  action_null_literal: 4
  future_null_literal: 43
  status_dist: {'complete': 56, 'active': 54}
  year_in_corpus: True
  candidate_meeting_count: 7
  candidate_segment_count: 29395
=== 2024 ===
  rows: 71
  target_issue_id_dupes: 0
  issue_text_missing: 0
  action_null_literal: 1
  future_null_literal: 27
  status_dist: {'complete': 37, 'active': 34}
  year_in_corpus: True
  candidate_meeting_count: 7
  candidate_segment_count: 49147

=== sample rows (target_issue_id / query_core / query_expanded / query_full trimmed) ===
-- 2020 (first 3) --
target_issue_id                                                   query_core                                               query_expanded                                                   query_full
  TGT_2020_0004                           코로나19로인해급변한상황을 고려하여5개년성과관리계획을 수정할것 코로나19로인해급변한상황을 고려하여5개년성과관리계획을 수정할것조치완료 코로나19 상황을고려한성과관리전략계획( 코로나19로인해급변한상황을 고려하여5개년성과관리계획을 수정할것조치완

## 15. Retrieval Smoke Test

**목적**: 지도 라벨을 만드는 것이 아니라, 현재 segment가 검색 문서로 "그럴듯하게" 작동하는지 육안 검토한다.
**입력**: `target_dfs`(2020/2024), `segment_df`(연도별 corpus 제한).
**출력**: 연도별 5개 target × Top 10 후보, `retrieval_smoke_candidates.csv`.
**가능한 실패**: `scikit-learn` 미설치 — 자동 설치하지 않고, **numpy 기반 수동 char/word TF-IDF + cosine
구현으로 대체**한다(00절에서 이미 `sklearn_available=False` 확인·기록됨). 이는 스킵이 아니라 명시적 대체이며
결과는 sklearn의 `TfidfVectorizer`와 동일한 공식(smooth idf, L2 정규화)을 따른다.


In [25]:
def char_ngrams(text, n_range):
    lo, hi = n_range
    grams = []
    for n in range(lo, hi + 1):
        grams.extend(text[i:i+n] for i in range(len(text) - n + 1))
    return grams

def word_ngrams(text, n_range):
    tokens = [t for t in re.split(r"\s+", text) if t]
    lo, hi = n_range
    grams = []
    for n in range(lo, hi + 1):
        for i in range(len(tokens) - n + 1):
            grams.append(" ".join(tokens[i:i+n]))
    return grams

def fit_tfidf_corpus(doc_texts: dict, analyzer_fn):
    """doc_texts: {doc_id: text} -> returns (doc_tf: {doc_id: Counter}, idf: {term: float}, doc_norm: {doc_id: float})"""
    doc_tf = {did: Counter(analyzer_fn(text)) for did, text in doc_texts.items()}
    df = Counter()
    for tf in doc_tf.values():
        df.update(tf.keys())
    N = len(doc_tf)
    idf = {term: (np.log((1 + N) / (1 + d)) + 1.0) for term, d in df.items()}
    doc_norm = {}
    for did, tf in doc_tf.items():
        sq = sum((cnt * idf[term]) ** 2 for term, cnt in tf.items())
        doc_norm[did] = np.sqrt(sq) if sq > 0 else 0.0
    return doc_tf, idf, doc_norm

def tfidf_cosine_scores(query_text, analyzer_fn, doc_tf, idf, doc_norm):
    q_tf = Counter(analyzer_fn(query_text))
    q_vec = {term: cnt * idf.get(term, 0.0) for term, cnt in q_tf.items()}
    q_norm = np.sqrt(sum(v * v for v in q_vec.values()))
    scores = {}
    if q_norm == 0:
        return {did: 0.0 for did in doc_tf}
    for did, tf in doc_tf.items():
        if doc_norm[did] == 0:
            scores[did] = 0.0
            continue
        dot = sum(q_vec[t] * (tf[t] * idf[t]) for t in q_vec if t in tf)
        scores[did] = dot / (q_norm * doc_norm[did])
    return scores

print("manual TF-IDF functions defined (char analyzer n-gram", CHAR_NGRAM_RANGE, ", word analyzer n-gram", WORD_NGRAM_RANGE, ")")


manual TF-IDF functions defined (char analyzer n-gram (3, 5) , word analyzer n-gram (1, 2) )


In [26]:
def pick_diverse_targets(df, n, seed):
    lengths = df["issue_text"].str.len()
    bins = pd.qcut(lengths, q=min(n, lengths.nunique()), duplicates="drop")
    picked = []
    rng_local = np.random.default_rng(seed)
    for _, grp in df.groupby(bins, observed=True):
        if len(grp) == 0:
            continue
        idx = rng_local.integers(0, len(grp))
        picked.append(grp.iloc[idx])
    picked_df = pd.DataFrame(picked)
    if len(picked_df) < n:
        remaining = df[~df.index.isin(picked_df.index)]
        extra = remaining.sample(min(n - len(picked_df), len(remaining)), random_state=seed)
        picked_df = pd.concat([picked_df, extra])
    return picked_df.head(n)

smoke_results = []
smoke_display_blocks = []

if not RUN_RETRIEVAL_SMOKE_TEST:
    print("RUN_RETRIEVAL_SMOKE_TEST is False -- section skipped by configuration")
else:
    for year, target_df in target_dfs.items():
        candidate_meetings = registry_df[registry_df["meeting_year"] == year[2:]]["meeting_id"].tolist()
        year_corpus = segment_df[segment_df["meeting_id"].isin(candidate_meetings) & segment_df["tfidf_ready"]].copy()
        if year_corpus.empty:
            print(f"{year}: no corpus segments available -- SKIP")
            continue
        doc_texts = dict(zip(year_corpus["segment_no"], year_corpus["normalized_text"]))

        char_tf, char_idf, char_norm = fit_tfidf_corpus(doc_texts, lambda t: char_ngrams(t, CHAR_NGRAM_RANGE))
        word_tf, word_idf, word_norm = fit_tfidf_corpus(doc_texts, lambda t: word_ngrams(t, WORD_NGRAM_RANGE))

        targets_picked = pick_diverse_targets(target_df, SMOKE_TARGETS_PER_YEAR, RANDOM_SEED)
        print(f"=== {year}: corpus_segments={len(year_corpus)}  targets_picked={len(targets_picked)} ===")

        for _, trow in targets_picked.iterrows():
            query = trow["query_core"]
            char_scores = tfidf_cosine_scores(query, lambda t: char_ngrams(t, CHAR_NGRAM_RANGE), char_tf, char_idf, char_norm)
            word_scores = tfidf_cosine_scores(query, lambda t: word_ngrams(t, WORD_NGRAM_RANGE), word_tf, word_idf, word_norm)
            combined = {did: CHAR_SCORE_WEIGHT * char_scores[did] + WORD_SCORE_WEIGHT * word_scores[did] for did in doc_texts}
            top = sorted(combined.items(), key=lambda kv: kv[1], reverse=True)[:SMOKE_TOP_K]

            block_lines = [f"[Target] issue_no={trow['issue_no']}  issue_text={query[:120]}"]
            for rank, (seg_no, score) in enumerate(top, start=1):
                seg_row = year_corpus[year_corpus["segment_no"] == seg_no].iloc[0]
                smoke_results.append({
                    "target_issue_id": trow["target_issue_id"], "issue_no": trow["issue_no"],
                    "query_core": query, "rank": rank, "segment_no": seg_no,
                    "meeting_id": seg_row["meeting_id"], "segment_type": seg_row["segment_type"],
                    "char_score": char_scores[seg_no], "word_score": word_scores[seg_no], "combined_score": score,
                    "speaker_names": seg_row["speaker_names"], "page_start_no": seg_row["page_start_no"],
                    "page_end_no": seg_row["page_end_no"], "segment_text_preview": seg_row["normalized_text"][:300],
                    "relevance_label": "", "review_note": "",
                })
                if rank <= 5:
                    block_lines.append(f"  [{rank}] meeting={seg_row['meeting_id']} speaker={seg_row['speaker_names']} "
                                        f"score={score:.4f} text={seg_row['normalized_text'][:150]}")
            smoke_display_blocks.append("\n".join(block_lines))

smoke_df = pd.DataFrame(smoke_results)
print()
print(f"total smoke candidate rows: {len(smoke_df)}")


=== 2020: corpus_segments=29395  targets_picked=5 ===


=== 2024: corpus_segments=49147  targets_picked=5 ===



total smoke candidate rows: 100


In [27]:
print("=== smoke test results (first 4 targets shown) ===\n")
for block in smoke_display_blocks[:4]:
    print(block)
    print()

SMOKE_CSV_PATH = OUTPUT_ROOT / "retrieval_smoke_candidates.csv"
smoke_df.to_csv(SMOKE_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"saved: {SMOKE_CSV_PATH}")

n_targets_tested = smoke_df["target_issue_id"].nunique() if len(smoke_df) else 0
print()
print("[SECTION 15 RESULT]")
print("status:", "PASS" if n_targets_tested > 0 else "SKIP")
print(f"targets_tested: {n_targets_tested}  candidate_rows: {len(smoke_df)}")
print("NOTE: results are NOT labeled as correct -- relevance_label/review_note columns left blank for human review")
print("next: Integrated quality gate")


=== smoke test results (first 4 targets shown) ===

[Target] issue_no=157  issue_text=호텔업재산세감면을위해 지자체와협의할것
  [1] meeting=050615 speaker=◯이용 위원  지자체는 몇 개 정도 되지요? | ◯한국관광공사사장 안영배  164개 지자체가 운 score=0.0490 text=◯이용 위원  지자체는 몇 개 정도 되지요?◯한국관광공사사장 안영배  164개 지자체가 운영하고 있습니다.
  [2] meeting=050615 speaker=◯이용 위원  지자체는 몇 개 정도 되지요? | ◯한국관광공사사장 안영배  164개 지자체가 운 score=0.0490 text=◯이용 위원  지자체는 몇 개 정도 되지요?◯한국관광공사사장 안영배  164개 지자체가 운영하고 있습니다.
  [3] meeting=050673 speaker=◯한국문화예술위원장 박종관  지자체에서 실시 score=0.0389 text=◯한국문화예술위원장 박종관  지자체에서 실시하는 건축물 미술작품 심의는 위원님께서 지적하신 대로 지자체 조례에 따라서 운영되는 곳이 많은데요. 말씀하시는 부분은 향후 문화체육관광부나 지자체 등 관계기관과 협의를 잘해서 지적해주신 문제점을 해소하도록 노력하겠습니다.감사합
  [4] meeting=050615 speaker=◯이용 위원  지자체는 몇 개 정도 되지요? score=0.0384 text=◯이용 위원  지자체는 몇 개 정도 되지요?
  [5] meeting=050673 speaker=◯이병훈 위원  예. | ◯한국문화예술위원장 박종관  지자체에서 실시 score=0.0384 text=◯이병훈 위원  예.◯한국문화예술위원장 박종관  지자체에서 실시하는 건축물 미술작품 심의는 위원님께서 지적하신 대로 지자체 조례에 따라서 운영되는 곳이 많은데요. 말씀하시는 부분은 향후 문화체육관광부나 지자체 등 관계기관과 협의를 잘해서 지적해주신 문제점을 해소하도록 

[Target] issu

## 16. 통합 Quality Gate

**목적**: 앞선 모든 절의 핵심 지표를 하나의 `audit_metrics` DataFrame으로 모으고, 최종 판정
(`READY_FOR_CANONICAL_SEGMENT_EXPORT` / `CONDITIONAL_READY` / `NOT_READY`)을 내린다.
**입력**: 00~15절에서 계산된 모든 지표 변수.
**출력**: `audit_metrics` DataFrame, `readiness_score`, `gate_decision`.
**가능한 실패**: 핵심(core) 지표 실패 1건 이상이면 `NOT_READY`, 경고만 있으면 `CONDITIONAL_READY`.


In [28]:
def metric_row(name, value, status, threshold, detail):
    return {"pipeline_run_id": RUN_STARTED_UTC.isoformat(), "metric_name": name, "metric_value": value,
            "threshold": threshold, "status": status, "detail": detail}

audit_metrics_rows = [
    metric_row("registry_pdf_integrity", pdf_integrity_summary["n_signature_fail"] + pdf_integrity_summary["n_open_fail"]
               + pdf_integrity_summary["n_page_count_mismatch"] + pdf_integrity_summary["n_sha256_mismatch"],
               "PASS" if registry_pdf_gate_pass else "FAIL", "==0", "sum of all PDF integrity failure types"),
    metric_row("block_pk_duplicates", block_pk_dupes, "PASS" if block_pk_dupes == 0 else "FAIL", "==0", ""),
    metric_row("turn_pk_duplicates", turn_pk_dupes, "PASS" if turn_pk_dupes == 0 else "FAIL", "==0", ""),
    metric_row("block_fk_orphans", block_fk_orphans, "PASS" if block_fk_orphans == 0 else "FAIL", "==0", ""),
    metric_row("page_coverage_ratio", page_coverage_ratio, "PASS" if page_coverage_ratio == 1.0 else "FAIL", "==1.0", ""),
    metric_row("block_unknown_ratio", unknown_ratio, "PASS" if unknown_ratio == 0 else "WARN", "==0", ""),
    metric_row("turn_inheritance_fail", metric_counts["inheritance_fail"], "PASS" if metric_counts["inheritance_fail"]==0 else "FAIL", "==0", ""),
    metric_row("turn_containment_fail", metric_counts["containment_fail"],
               "PASS" if metric_counts["containment_fail"] == 0 else "WARN", "==0",
               "known root cause: EMPTY-block accounting gap, 2/65590 turns"),
    metric_row("turn_adjacency_fail", metric_counts["adjacency_fail"], "PASS" if metric_counts["adjacency_fail"]==0 else "FAIL", "==0", ""),
    metric_row("turn_atomicity_fail", metric_counts["atomicity_fail"], "PASS" if metric_counts["atomicity_fail"]==0 else "FAIL", "==0", ""),
    metric_row("turn_n_blocks_mismatch", metric_counts["n_blocks_mismatch"],
               "PASS" if metric_counts["n_blocks_mismatch"] == 0 else "WARN", "==0", "same EMPTY-block root cause"),
    metric_row("multiple_speaker_headers", metric_counts["multiple_speaker_headers"], "PASS" if metric_counts["multiple_speaker_headers"]==0 else "FAIL", "==0", ""),
    metric_row("speaker_turn_without_header", metric_counts["speaker_turn_without_header"], "PASS" if metric_counts["speaker_turn_without_header"]==0 else "FAIL", "==0", ""),
    metric_row("orphan_turn_ratio", orphan_ratio, "PASS" if orphan_ratio < 0.01 else "WARN", "<0.01", "42 orphan front-matter turns / 65590 expected"),
    metric_row("empty_speaker_turn_ratio", empty_speaker_turn_ratio, "PASS" if empty_speaker_turn_ratio == 0 else "WARN", "==0", ""),
    metric_row("segment_expected_vs_observed", total_observed - total_expected, "PASS" if total_observed == total_expected else "FAIL", "==0", ""),
    metric_row("segment_fk_orphans", sum(fk_orphans.values()), "PASS" if sum(fk_orphans.values()) == 0 else "FAIL", "==0", ""),
    metric_row("segment_assembly_fail", assembly_fail_count, "PASS" if assembly_fail_count == 0 else "FAIL", "==0", ""),
    metric_row("segment_traceability_rate", traceability_rate, "PASS" if traceability_rate == 1.0 else "WARN", "==1.0",
               "gap matches the 2 known EMPTY-block turns"),
    metric_row("target_year_compatibility", int(all(s["year_in_corpus"] for s in target_compat_summary.values())),
               "PASS" if all(s["year_in_corpus"] for s in target_compat_summary.values()) else "FAIL", "==1", ""),
    metric_row("retrieval_smoke_targets_tested", n_targets_tested, "PASS" if n_targets_tested >= 2*SMOKE_TARGETS_PER_YEAR - 2 else "WARN",
               f">={2*SMOKE_TARGETS_PER_YEAR}", ""),
    metric_row("canonical_pages_index_export_present", 0, "INFO", "n/a",
               "pages.parquet/index_map.parquet/retrieval_segments.parquet not produced -- audit-only run by "
               "design (this metric is NOT a Phase 1 pass criterion; it becomes Phase 2's job once Phase 1 gates green)"),
    metric_row("sklearn_available", int(_sklearn_ok), "INFO", "n/a",
               "smoke test substitutes a manual numpy TF-IDF when sklearn is absent -- does not affect canonical readiness"),
]
audit_metrics = pd.DataFrame(audit_metrics_rows)

CORE_METRICS = {
    "registry_pdf_integrity", "block_pk_duplicates", "turn_pk_duplicates", "block_fk_orphans",
    "page_coverage_ratio", "turn_inheritance_fail", "turn_adjacency_fail", "turn_atomicity_fail",
    "multiple_speaker_headers", "speaker_turn_without_header", "segment_expected_vs_observed",
    "segment_fk_orphans", "segment_assembly_fail", "target_year_compatibility",
}
core_rows = audit_metrics[audit_metrics["metric_name"].isin(CORE_METRICS)]
core_fail_count = int((core_rows["status"] == "FAIL").sum())
warning_count = int((audit_metrics["status"] == "WARN").sum())
pass_count = int((audit_metrics["status"] == "PASS").sum())

readiness_score = round(pass_count / len(audit_metrics), 4)

if core_fail_count > 0:
    gate_decision = "NOT_READY"
elif warning_count > 0:
    gate_decision = "CONDITIONAL_READY"
else:
    gate_decision = "READY_FOR_CANONICAL_SEGMENT_EXPORT"

print(audit_metrics.to_string(index=False))
print()
print(f"core_fail_count: {core_fail_count}")
print(f"warning_count: {warning_count}")
print(f"pass_count: {pass_count} / {len(audit_metrics)}")
print(f"readiness_score: {readiness_score}")
print(f"gate_decision: {gate_decision}")

print()
print("[SECTION 16 RESULT]")
print("status: PASS (gate computed)")
print(f"gate_decision: {gate_decision}")
print(f"core_fail_count: {core_fail_count}  warning_count: {warning_count}  readiness_score: {readiness_score}")
print("next: Audit exports (parquet/csv/manifest/report)")


                 pipeline_run_id                          metric_name  metric_value threshold status                                                                                                                                                                                               detail
2026-07-21T08:27:23.406661+00:00               registry_pdf_integrity       0.00000       ==0   PASS                                                                                                                                                               sum of all PDF integrity failure types
2026-07-21T08:27:23.406661+00:00                  block_pk_duplicates       0.00000       ==0   PASS                                                                                                                                                                                                     
2026-07-21T08:27:23.406661+00:00                   turn_pk_duplicates       0.00000       ==0   PASS      

## 17. Audit 산출물 · Manifest · Report

**목적**: 이 감사의 모든 결과를 `outputs/segment_validation_audit/`에 저장한다. 여기서 나온 어떤 파일도
canonical SSOT 산출물이 아니다.
**입력**: 이전 모든 절의 metric·anomaly 변수.
**출력**: `audit_metrics.parquet`, `audit_anomalies.parquet`, `turn_manual_review_sample.csv`(10절에서 이미 저장),
`segment_manual_review_sample.csv`, `retrieval_smoke_candidates.csv`(15절에서 이미 저장),
`SEGMENT_VALIDATION_AUDIT.md`, `audit_manifest.json`.
**가능한 실패**: 디스크 쓰기 실패, 재로딩 시 row count 불일치.


In [29]:
anomaly_records = []

def add_anomaly(category, entity_type, entity_id, meeting_id, issue_type, detail, severity):
    anomaly_records.append({"category": category, "entity_type": entity_type, "entity_id": entity_id,
                             "meeting_id": meeting_id, "issue_type": issue_type, "detail": detail, "severity": severity})

for turn_no in mismatched_turns:
    add_anomaly("turn_hierarchy", "turn", turn_no, joined.loc[turn_no, "meeting_id"],
                "n_blocks_mismatch_or_containment_fail", "EMPTY-block accounting gap (see Section 08 diagnosis)", "WARN")

for _, row in long_turn_candidates.iterrows():
    add_anomaly("turn_distribution", "turn", row["turn_no"], row["meeting_id"], "long_turn_candidate",
                f"char_count={row['char_count']} n_blocks={row['n_blocks']}", "INFO")
for _, row in spacing_collapse_candidates.iterrows():
    add_anomaly("text_quality", "turn", row["turn_no"], row["meeting_id"], "spacing_collapse_suspect",
                f"char_count={row['char_count']}", "WARN")
for _, row in over_merge_candidates.iterrows():
    add_anomaly("turn_distribution", "turn", row["turn_no"], row["meeting_id"], "over_merge_candidate",
                f"n_blocks={row['n_blocks']} char_count={row['char_count']}", "INFO")
for _, row in over_split_candidates.iterrows():
    add_anomaly("turn_distribution", "turn", row["turn_no"], row["meeting_id"], "over_split_candidate",
                f"char_count={row['char_count']}", "INFO")
if len(invalid_sample):
    for _, row in invalid_sample.iterrows():
        add_anomaly("segment_integrity", "segment", row["segment_no"], row["meeting_id"], "assembly_mismatch", "", "FAIL")

audit_anomalies = pd.DataFrame(anomaly_records)
print(f"total anomaly records: {len(audit_anomalies)}")
print(audit_anomalies["category"].value_counts().to_dict() if len(audit_anomalies) else "(none)")


total anomaly records: 2474
{'text_quality': 1703, 'turn_distribution': 771}


In [30]:
# stratified segment manual review sample (parallel to Section 10's turn sample)
seg_used_idx = set()
seg_groups = []

def sample_seg_group(df, name, n, exclude_idx):
    pool = df[~df.index.isin(exclude_idx)]
    n = min(n, len(pool))
    if n == 0:
        return pd.DataFrame()
    picked = pool.sample(n, random_state=RANDOM_SEED).copy()
    picked["review_group"] = name
    return picked

for seg_type in ["SEG_TURN", "SEG_PREV_CURR", "SEG_CURR_NEXT"]:
    pool = segment_df[segment_df["segment_type"] == seg_type]
    g = sample_seg_group(pool, f"type_{seg_type}", 6, seg_used_idx)
    seg_used_idx |= set(g.index); seg_groups.append(g)

seg_char_p95 = seg_char_q["p95"]
g = sample_seg_group(segment_df[segment_df["char_count"] >= seg_char_p95], "long_segment", 6, seg_used_idx)
seg_used_idx |= set(g.index); seg_groups.append(g)
g = sample_seg_group(needs_chunk, "needs_chunk", 6, seg_used_idx); seg_used_idx |= set(g.index); seg_groups.append(g)
g = sample_seg_group(segment_df[segment_df["anchor_is_procedural_or_orphan"]], "procedural_or_orphan_anchor", 6, seg_used_idx)
seg_used_idx |= set(g.index); seg_groups.append(g)
g = sample_seg_group(segment_df[segment_df["normalized_text"].duplicated(keep=False)], "duplicate_text", 5, seg_used_idx)
seg_used_idx |= set(g.index); seg_groups.append(g)
g = sample_seg_group(segment_df, "random_normal", 8, seg_used_idx); seg_used_idx |= set(g.index); seg_groups.append(g)

segment_review_sample = pd.concat([g for g in seg_groups if len(g)], ignore_index=True)
segment_review_sample["suspected_issue"] = segment_review_sample["review_group"]
segment_review_sample["manual_label"] = ""
segment_review_sample["review_note"] = ""
SEG_REVIEW_COLUMNS = ["review_group", "segment_no", "meeting_id", "segment_type", "anchor_turn_no",
                      "page_start_no", "page_end_no", "char_count", "speaker_names", "normalized_text",
                      "suspected_issue", "manual_label", "review_note"]
segment_review_sample_out = segment_review_sample[SEG_REVIEW_COLUMNS]

SEG_REVIEW_CSV_PATH = OUTPUT_ROOT / "segment_manual_review_sample.csv"
segment_review_sample_out.to_csv(SEG_REVIEW_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"segment review sample rows: {len(segment_review_sample_out)}  groups: {segment_review_sample_out['review_group'].nunique()}")
print(f"saved: {SEG_REVIEW_CSV_PATH}")


segment review sample rows: 49  groups: 8
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_validation_audit/segment_manual_review_sample.csv


In [31]:
METRICS_PARQUET_PATH = OUTPUT_ROOT / "audit_metrics.parquet"
ANOMALIES_PARQUET_PATH = OUTPUT_ROOT / "audit_anomalies.parquet"

audit_metrics.to_parquet(METRICS_PARQUET_PATH, index=False)
audit_anomalies.to_parquet(ANOMALIES_PARQUET_PATH, index=False)
print(f"saved: {METRICS_PARQUET_PATH} ({len(audit_metrics)} rows)")
print(f"saved: {ANOMALIES_PARQUET_PATH} ({len(audit_anomalies)} rows)")

RUN_FINISHED_UTC = datetime.now(timezone.utc)

audit_manifest = {
    "project": "P3_CULTURE",
    "stage": "TURN_TO_RETRIEVAL_SEGMENT_AUDIT",
    "notebook": str(NOTEBOOK_PATH.relative_to(PROJECT_ROOT)),
    "run_started_utc": RUN_STARTED_UTC.isoformat(),
    "run_finished_utc": RUN_FINISHED_UTC.isoformat(),
    "git_branch": GIT_BRANCH, "git_head": GIT_HEAD,
    "source_mode": SOURCE_MODE,
    "registry_rows": int(len(registry_df)),
    "pdf_files": int(pdf_integrity_summary["n_files"]),
    "block_rows": int(len(block_df)),
    "turn_rows": int(len(turn_df)),
    "segment_rows": int(len(segment_df)),
    "block_pk_failures": block_pk_dupes,
    "turn_pk_failures": turn_pk_dupes,
    "block_fk_failures": block_fk_orphans,
    "turn_fk_failures": turn_meeting_fk_orphans,
    "containment_failures": metric_counts["containment_fail"],
    "adjacency_failures": metric_counts["adjacency_fail"],
    "atomicity_failures": metric_counts["atomicity_fail"],
    "n_blocks_mismatch": metric_counts["n_blocks_mismatch"],
    "long_turn_candidates": int(len(long_turn_candidates)),
    "spacing_collapse_candidates": int(len(spacing_collapse_candidates)),
    "orphan_turn_ratio": orphan_ratio,
    "empty_speaker_turn_ratio": empty_speaker_turn_ratio,
    "expected_segment_count": total_expected,
    "observed_segment_count": total_observed,
    "segment_fk_failures": sum(fk_orphans.values()),
    "traceability_rate": traceability_rate,
    "target_rows_2020": target_compat_summary["2020"]["rows"],
    "target_rows_2024": target_compat_summary["2024"]["rows"],
    "retrieval_smoke_candidates": int(len(smoke_df)),
    "core_fail_count": core_fail_count,
    "warning_count": warning_count,
    "readiness_score": readiness_score,
    "gate_decision": gate_decision,
    "canonical_exported": False,
    "outputs": {
        "audit_report": "outputs/segment_validation_audit/SEGMENT_VALIDATION_AUDIT.md",
        "metrics_parquet": "outputs/segment_validation_audit/audit_metrics.parquet",
        "anomalies_parquet": "outputs/segment_validation_audit/audit_anomalies.parquet",
        "turn_review_csv": "outputs/segment_validation_audit/turn_manual_review_sample.csv",
        "segment_review_csv": "outputs/segment_validation_audit/segment_manual_review_sample.csv",
        "smoke_test_csv": "outputs/segment_validation_audit/retrieval_smoke_candidates.csv",
    },
    "main_blockers": (
        ([f"EMPTY-type block accounting gap affects {metric_counts['containment_fail']}/{len(turn_df)} turns "
          "(containment/n_blocks) -- fix build_block_and_turn_df before canonical export"]
         if metric_counts["containment_fail"] > 0 or metric_counts["n_blocks_mismatch"] > 0 else [])
        + (["scikit-learn not installed -- smoke test used a manual numpy TF-IDF substitute, not sklearn's TfidfVectorizer"]
           if not _sklearn_ok else [])
    ),
    "next_required_action": (
        "Fix EMPTY-block accounting in the block/turn builder, then re-run this audit."
        if (metric_counts["containment_fail"] > 0 or metric_counts["n_blocks_mismatch"] > 0)
        else "Phase 1 gate is green (READY_FOR_CANONICAL_SEGMENT_EXPORT). Proceed to Phase 2: build the canonical "
             "05~10 production notebooks (05_pdf_page_block_export, 06_index_block_classification, "
             "07_speaker_turn_export, 08_retrieval_segment_export, 09_etl_quality_integration) in SEPARATE notebooks "
             "from this audit -- do not just rename this notebook's in-memory DataFrames to canonical parquet."
    ),
}

MANIFEST_PATH = OUTPUT_ROOT / "audit_manifest.json"
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(audit_manifest, f, ensure_ascii=False, indent=2, default=str)
print(f"saved: {MANIFEST_PATH}")


saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_validation_audit/audit_metrics.parquet (23 rows)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_validation_audit/audit_anomalies.parquet (2474 rows)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_validation_audit/audit_manifest.json


In [32]:
report_lines = []
report_lines.append("# SEGMENT_VALIDATION_AUDIT")
report_lines.append("")
report_lines.append(f"- run: {RUN_STARTED_UTC.isoformat()} ~ {RUN_FINISHED_UTC.isoformat()}")
report_lines.append(f"- git: `{GIT_BRANCH}` @ `{GIT_HEAD}`")
report_lines.append(f"- source_mode: **{SOURCE_MODE}**")
report_lines.append(f"- gate_decision: **{gate_decision}**  (readiness_score={readiness_score}, "
                     f"core_fail={core_fail_count}, warnings={warning_count})")
report_lines.append("")
report_lines.append("## 규모")
report_lines.append(f"- registry_rows={len(registry_df)}  pdf_files={pdf_integrity_summary['n_files']}")
report_lines.append(f"- block_rows={len(block_df)}  turn_rows={len(turn_df)}  segment_rows={len(segment_df)}")
report_lines.append("")
report_lines.append("## 핵심 발견")
report_lines.append(f"- PK/FK/page coverage: block_pk_dup={block_pk_dupes}, turn_pk_dup={turn_pk_dupes}, "
                     f"block_fk_orphan={block_fk_orphans}, page_coverage_ratio={page_coverage_ratio}")
_hierarchy_note = (
    "(원인: EMPTY-type block이 n_blocks/block_end_no 갱신에서 누락되는 버그 -- Phase 1에서 수정 완료, 0건)"
    if metric_counts["containment_fail"] == 0 and metric_counts["n_blocks_mismatch"] == 0
    else "(EMPTY-type block 회계 버그 미해결 -- build_block_and_turn_df 수정 필요)"
)
report_lines.append(f"- turn 위계 전수 감사: containment_fail={metric_counts['containment_fail']}, "
                     f"n_blocks_mismatch={metric_counts['n_blocks_mismatch']} {_hierarchy_note}")
report_lines.append(f"- segment: expected={total_expected} observed={total_observed} (일치), "
                     f"assembly_fail={assembly_fail_count}, traceability_rate={traceability_rate:.6f}")
report_lines.append(f"- target 호환성: 2020={target_compat_summary['2020']['rows']}행, "
                     f"2024={target_compat_summary['2024']['rows']}행, 둘 다 corpus 연도 존재")
report_lines.append(f"- retrieval smoke test: {n_targets_tested}개 target × top {SMOKE_TOP_K} "
                     "(scikit-learn 미설치로 수동 numpy TF-IDF 대체 구현 사용)")
report_lines.append("")
report_lines.append("## audit_metrics 요약")
report_lines.append("```text")
report_lines.append(audit_metrics[["metric_name", "metric_value", "status"]].to_string(index=False))
report_lines.append("```")
report_lines.append("")
report_lines.append("## 다음 필요 조치")
for b in audit_manifest["main_blockers"]:
    report_lines.append(f"- {b}")
report_lines.append("")
report_lines.append(f"**최종 판정: {gate_decision}**")

REPORT_PATH = OUTPUT_ROOT / "SEGMENT_VALIDATION_AUDIT.md"
REPORT_PATH.write_text("\n".join(str(l) for l in report_lines), encoding="utf-8")
print(f"saved: {REPORT_PATH}")

print()
print("[SECTION 17 RESULT]")
print("status: PASS")
print(f"outputs_written: metrics={METRICS_PARQUET_PATH.name}, anomalies={ANOMALIES_PARQUET_PATH.name}, "
      f"turn_review={REVIEW_CSV_PATH.name}, segment_review={SEG_REVIEW_CSV_PATH.name}, "
      f"smoke={SMOKE_CSV_PATH.name}, manifest={MANIFEST_PATH.name}, report={REPORT_PATH.name}")
print("next: reload verification (see final terminal summary)")


saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_validation_audit/SEGMENT_VALIDATION_AUDIT.md

[SECTION 17 RESULT]
status: PASS
outputs_written: metrics=audit_metrics.parquet, anomalies=audit_anomalies.parquet, turn_review=turn_manual_review_sample.csv, segment_review=segment_manual_review_sample.csv, smoke=retrieval_smoke_candidates.csv, manifest=audit_manifest.json, report=SEGMENT_VALIDATION_AUDIT.md
next: reload verification (see final terminal summary)


## 재로딩 검증 (self-check)

방금 저장한 산출물을 다시 읽어서 row count가 메모리 상 값과 일치하는지 확인한다.


In [33]:
_reload_checks = {}
_m = pd.read_parquet(METRICS_PARQUET_PATH)
_reload_checks["audit_metrics"] = (len(_m) == len(audit_metrics))
_a = pd.read_parquet(ANOMALIES_PARQUET_PATH)
_reload_checks["audit_anomalies"] = (len(_a) == len(audit_anomalies))
_t = pd.read_csv(REVIEW_CSV_PATH, encoding="utf-8-sig")
_reload_checks["turn_review_csv"] = (len(_t) == len(review_sample_out))
_s = pd.read_csv(SEG_REVIEW_CSV_PATH, encoding="utf-8-sig")
_reload_checks["segment_review_csv"] = (len(_s) == len(segment_review_sample_out))
_sm = pd.read_csv(SMOKE_CSV_PATH, encoding="utf-8-sig")
_reload_checks["smoke_csv"] = (len(_sm) == len(smoke_df))
with open(MANIFEST_PATH, encoding="utf-8") as f:
    _mf = json.load(f)
_reload_checks["manifest_json"] = (_mf["turn_rows"] == len(turn_df))

print("reload checks:", _reload_checks)
assert all(_reload_checks.values()), "reload verification failed -- exported file row counts do not match in-memory data"
print("all reload checks passed")


reload checks: {'audit_metrics': True, 'audit_anomalies': True, 'turn_review_csv': True, 'segment_review_csv': True, 'smoke_csv': True, 'manifest_json': True}
all reload checks passed


## 최종 터미널 요약


In [34]:
print("# PDF Segment Validation Audit Result")
print()
print(f"- project_root: {PROJECT_ROOT}")
print(f"- notebook: {NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}")
print(f"- source_mode: {SOURCE_MODE}")
print(f"- registry_rows: {len(registry_df)}")
print(f"- pdf_files: {pdf_integrity_summary['n_files']}")
print(f"- block_rows: {len(block_df)}")
print(f"- turn_rows: {len(turn_df)}")
print(f"- segment_rows: {len(segment_df)}")
print(f"- block_pk_failures: {block_pk_dupes}")
print(f"- turn_pk_failures: {turn_pk_dupes}")
print(f"- block_fk_failures: {block_fk_orphans}")
print(f"- turn_fk_failures: {turn_meeting_fk_orphans}")
print(f"- containment_failures: {metric_counts['containment_fail']}")
print(f"- adjacency_failures: {metric_counts['adjacency_fail']}")
print(f"- atomicity_failures: {metric_counts['atomicity_fail']}")
print(f"- long_turn_candidates: {len(long_turn_candidates)}")
print(f"- spacing_collapse_candidates: {len(spacing_collapse_candidates)}")
print(f"- orphan_turn_ratio: {orphan_ratio}")
print(f"- empty_speaker_turn_ratio: {empty_speaker_turn_ratio}")
print(f"- expected_segment_count: {total_expected}")
print(f"- observed_segment_count: {total_observed}")
print(f"- segment_fk_failures: {sum(fk_orphans.values())}")
print(f"- traceability_rate: {traceability_rate}")
print(f"- target_rows_2020: {target_compat_summary['2020']['rows']}")
print(f"- target_rows_2024: {target_compat_summary['2024']['rows']}")
print(f"- retrieval_smoke_candidates: {len(smoke_df)}")
print(f"- core_fail_count: {core_fail_count}")
print(f"- warning_count: {warning_count}")
print(f"- readiness_score: {readiness_score}")
print(f"- gate_decision: {gate_decision}")
print(f"- audit_report: {REPORT_PATH}")
print(f"- metrics_parquet: {METRICS_PARQUET_PATH}")
print(f"- anomalies_parquet: {ANOMALIES_PARQUET_PATH}")
print(f"- turn_review_csv: {REVIEW_CSV_PATH}")
print(f"- segment_review_csv: {SEG_REVIEW_CSV_PATH}")
print(f"- smoke_test_csv: {SMOKE_CSV_PATH}")
print(f"- main_blockers: {audit_manifest['main_blockers']}")
print(f"- next_required_action: {audit_manifest['next_required_action']}")
print()
print(json.dumps({
    "project": "P3_CULTURE", "stage": "TURN_TO_RETRIEVAL_SEGMENT_AUDIT", "source_mode": SOURCE_MODE,
    "registry_rows": len(registry_df), "pdf_files": pdf_integrity_summary["n_files"],
    "block_rows": len(block_df), "turn_rows": len(turn_df), "segment_rows": len(segment_df),
    "block_pk_failures": block_pk_dupes, "turn_pk_failures": turn_pk_dupes,
    "block_fk_failures": block_fk_orphans, "turn_fk_failures": turn_meeting_fk_orphans,
    "containment_failures": metric_counts["containment_fail"], "adjacency_failures": metric_counts["adjacency_fail"],
    "atomicity_failures": metric_counts["atomicity_fail"], "orphan_turn_ratio": orphan_ratio,
    "empty_speaker_turn_ratio": empty_speaker_turn_ratio, "traceability_rate": traceability_rate,
    "target_rows": target_compat_summary["2020"]["rows"] + target_compat_summary["2024"]["rows"],
    "retrieval_smoke_pairs": len(smoke_df), "core_fail_count": core_fail_count, "warning_count": warning_count,
    "readiness_score": readiness_score, "gate_decision": gate_decision, "canonical_exported": False,
    "next_required_action": audit_manifest["next_required_action"], "blockers": audit_manifest["main_blockers"],
}, ensure_ascii=False, indent=2))


# PDF Segment Validation Audit Result

- project_root: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE
- notebook: 91_pdf_segment_validation_audit.ipynb
- source_mode: rebuild_from_pdf
- registry_rows: 42
- pdf_files: 42
- block_rows: 293717
- turn_rows: 65590
- segment_rows: 196686
- block_pk_failures: 0
- turn_pk_failures: 0
- block_fk_failures: 0
- turn_fk_failures: 0
- containment_failures: 0
- adjacency_failures: 0
- atomicity_failures: 0
- long_turn_candidates: 656
- spacing_collapse_candidates: 1703
- orphan_turn_ratio: 0.0006403415154749199
- empty_speaker_turn_ratio: 0.0
- expected_segment_count: 196686
- observed_segment_count: 196686
- segment_fk_failures: 0
- traceability_rate: 1.0
- target_rows_2020: 110
- target_rows_2024: 71
- retrieval_smoke_candidates: 100
- core_fail_count: 0
- warning_count: 0
- readiness_score: 0.913
- gate_decision: READY_FOR_CANONICAL_SEGMENT_EXPORT
- audit_report: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/outputs/segment_val